# Del clima a la generación real: plantas solares de Colombia

Pregunta: ¿cuánto de la variación diaria de la generación de una planta solar colombiana se explica con el clima de ese día?

El cuaderno baja la generación real desde XM, la empareja con las plantas del dataset de Mantilla-Guerra et al. (2026), trae el clima
de cada sitio y compara varios modelos. Cada decisión lleva su fuente. Las referencias están al final y en `referencias.bib`, con los DOI
verificados contra Crossref.

Por qué no usamos `solar_aptitude`: el índice del dataset sale solo de pendiente, orientación, sombra y curvatura (Ec. 1 del paper), así que
el clima no puede predecirlo. El factor de capacidad real sí depende del clima por física, y eso se puede medir.

## Pipeline

De la generación real de XM y el clima de Open-Meteo a la base diaria de Colombia.

![Pipeline](fig_pipeline_clima.png)

## 1. Fuentes y corte de tiempo

| Fuente | Qué aporta | Verificación |
|---|---|---|
| XM, API pública `servapibi.xm.com.co` (XM, 2026) | Generación horaria por planta (`Gene`), capacidad efectiva neta diaria (`CapEfecNeta`) | Se probó la API sin clave. La licencia de los datos no se verificó |
| Dataset del paper, con la tabla de GEM de febrero de 2026 (Mantilla-Guerra et al., 2026; GEM, 2026) | Coordenadas, nombre y capacidad de cada planta | DOI de la versión en revista verificado en Crossref |
| Open-Meteo, API histórica (Zippenfenig, 2023) | Radiación, nubosidad, temperatura, humedad, viento y radiación en plano inclinado, por hora | La documentación dice que el modelo por defecto combina IFS HRES, ERA5 y ERA5-Land (Hersbach et al., 2020; Muñoz-Sabater et al., 2021) |

Corte de tiempo: el paper usa el release de GEM de febrero de 2026, el potencial de Global Solar Atlas del período 2018-2024 y carreteras de
OSM del 14 de marzo de 2026. No dice qué años promedia ERA5. La ventana de este cuaderno termina el 28 de febrero de 2026, el mes del release
de GEM, y empieza el 1 de enero de 2024. El inicio es decisión nuestra.

El clima no es una medición en el sitio. Open-Meteo entrega el valor de la celda de un modelo. Comparamos dos productos con la misma consulta:
`era5`, de 0.25° y documentado en la literatura, y `best_match`, el valor por defecto, que en las pruebas cayó en una celda más fina (ver sección 6).

In [ ]:
import json, re, time, unicodedata, difflib, warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr, wilcoxon
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")

SEED = 42
RNG = np.random.default_rng(SEED)
START, END = date(2024, 1, 1), date(2026, 2, 28)
CAP_REF_DATE = "2026-02-15"      # fecha para decidir qué plantas pasan de 1 MW
MIN_MW = 1.0
RAMP_DAYS = 60                   # días de puesta en marcha que se excluyen en el análisis principal
MIN_DIAS = 100                   # una planta con menos días útiles que esto no entra a la evaluación
CLUSTER_KM = 100                 # plantas a menos de esta distancia comparten clima y se tratan como un grupo
XM_URL = "https://servapibi.xm.com.co/"
OM_URL = "https://archive-api.open-meteo.com/v1/archive"
DATA = Path("datos_xm")
DATA.mkdir(exist_ok=True)
GEM_CSV = Path("..") / "Dataset_Mundial_Final(2).csv"

## 2. Generación real desde XM

La API acepta rangos de hasta 30 días en datos horarios y diarios. Cada descarga se guarda en `datos_xm/` y, si el archivo ya existe, no se
vuelve a pedir.

Primero se listan los recursos de XM con fuente solar. Nos quedamos con los que tenían al menos 1 MW de capacidad efectiva neta en la fecha
de referencia. Los autogeneradores pequeños (`AGPE`) quedan fuera por tamaño.

In [ ]:
def xm_post(endpoint, body, tries=3):
    for _ in range(tries):
        try:
            r = requests.post(XM_URL + endpoint, json=body, timeout=180)
            if r.ok:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError(f"XM no respondió para {endpoint} {body.get('MetricId')}")

path_rec = DATA / "recursos_solares.csv"
if path_rec.exists():
    solares = pd.read_csv(path_rec)
else:
    lista = xm_post("lists", {"MetricId": "ListadoRecursos"})
    todos = pd.DataFrame([it["ListEntities"][0]["Values"] for it in lista["Items"]])
    es_solar = todos.EnerSource.str.contains("RAD|SOL", na=False) | todos.Type.str.contains("SOLAR", case=False, na=False)
    solares = todos[es_solar].copy()
    solares.to_csv(path_rec, index=False)

print(f"Recursos solares registrados en XM: {len(solares)}")
print("Autogeneradores o generación distribuida (AGPE, GD):", int(solares.Name.str.match(r"^(AGPE|GD) ").sum()))

In [ ]:
def capacidad(ini, fin, codigos):
    j = xm_post("daily", {"MetricId": "CapEfecNeta", "StartDate": str(ini), "EndDate": str(fin),
                          "Entity": "Recurso", "Filter": codigos})
    return [{"date": it["Date"], "code": e["Code"], "cap_MW": float(e["Value"]) / 1000}   # la API entrega kW
            for it in j["Items"] for e in it["DailyEntities"] if e.get("Value") not in (None, "")]

ref = pd.DataFrame(capacidad(CAP_REF_DATE, CAP_REF_DATE, solares.Code.tolist()))
codigos = sorted(ref[ref.cap_MW >= MIN_MW].code)
print(f"Plantas con al menos {MIN_MW} MW al {CAP_REF_DATE}: {len(codigos)}")

In [ ]:
def descarga_por_tramos(ruta, pedir):
    if ruta.exists():
        return pd.read_csv(ruta)
    filas, ini = [], START
    while ini <= END:
        fin = min(ini + timedelta(days=29), END)
        filas.extend(pedir(ini, fin))
        ini = fin + timedelta(days=1)
    df = pd.DataFrame(filas)
    df.to_csv(ruta, index=False)
    return df

def pedir_generacion(ini, fin):
    j = xm_post("hourly", {"MetricId": "Gene", "StartDate": str(ini), "EndDate": str(fin),
                           "Entity": "Recurso", "Filter": codigos})
    return [{"date": it["Date"], **ent["Values"]} for it in j["Items"] for ent in it["HourlyEntities"]]

gen = descarga_por_tramos(DATA / "generacion_horaria.csv", pedir_generacion)
cap_dia = descarga_por_tramos(DATA / "capacidad_diaria.csv", lambda a, b: capacidad(a, b, codigos))

horas = [c for c in gen.columns if c.startswith("Hour")]
gen[horas] = gen[horas].apply(pd.to_numeric, errors="coerce")
gen["date"] = pd.to_datetime(gen["date"])
gen["kwh"] = gen[horas].sum(axis=1)
cap_dia["date"] = pd.to_datetime(cap_dia["date"])
print("Generación:", gen.shape, "| capacidad diaria:", cap_dia.shape, "| plantas:", gen.code.nunique())

## 3. Factor de capacidad diario

El factor de capacidad (FC) es la energía del día dividida entre la que habría entregado la planta a potencia nominal las 24 horas:

`FC = kWh del día / (capacidad efectiva neta en kW × 24)`

La capacidad es la de ese mismo día, porque varias plantas crecen por etapas. Un revisor señaló que con la capacidad del día 15 de cada mes
el nivel de algunas plantas se movía por el denominador. Un FC mayor a 1.05 se descarta como dato roto.

Puesta en marcha: en una planta cuyo primer día con generación cae dentro de la ventana, los primeros `RAMP_DAYS` días se excluyen del análisis
principal. El criterio no mira ni la radiación ni el FC. Una planta que ya operaba el 1 de enero de 2024 no tiene su arranque observado y no pierde días.

In [ ]:
diario = gen[["code", "date", "kwh"]].merge(cap_dia, on=["code", "date"], how="inner")
diario["fc"] = diario.kwh / 1000 / (diario.cap_MW * 24)
diario = diario[(diario.kwh > 0) & (diario.cap_MW > 0)].copy()

primer = diario.groupby("code").date.min().rename("primer_dia")
diario = diario.merge(primer, on="code")
arranque_observado = diario.primer_dia > pd.Timestamp(START) + pd.Timedelta(days=7)
diario["dias_desde_arranque"] = np.where(arranque_observado, (diario.date - diario.primer_dia).dt.days, 10_000)

print("Filas planta-día con generación y capacidad:", len(diario))
print("Días con FC mayor a 1.05:", int((diario.fc > 1.05).sum()), "(se descartan)")
diario = diario[diario.fc <= 1.05]
print("Plantas con arranque observado dentro de la ventana:", int(diario.groupby("code").dias_desde_arranque.min().lt(10_000).sum()))

resumen = (diario.groupby("code")
           .agg(primer_dia=("date", "min"), dias=("date", "nunique"), cap_MW=("cap_MW", "median"), fc_medio=("fc", "mean"))
           .reset_index()
           .merge(solares[["Code", "Name"]], left_on="code", right_on="Code").drop(columns="Code"))
resumen.sort_values("cap_MW", ascending=False).round(3)

## 4. Emparejar con el dataset por nombre y capacidad

XM no entrega coordenadas y el clima las necesita. El dataset del paper sí las trae para algunas plantas. Primero se calculan los tres candidatos más parecidos de cada
planta en ese dataset. Luego se deja una tabla de decisiones hechas a mano, con el motivo de cada una. Siete plantas no aparecen en el dataset con un nombre reconocible,
y para ellas se usa la ficha pública de GEM, que es la misma fuente de la que sale el dataset. Cada una se contrastó con prensa o con el desarrollador.

Reglas para aceptar un emparejamiento:
- El nombre normalizado coincide de forma clara.
- La capacidad es igual o muy parecida. Dos casos tienen capacidad distinta y quedan anotados.
- Si hay dos filas de GEM con el mismo nombre, se verifica con una fuente externa cuál corresponde a la planta que genera.
- Si el nombre se parece pero la capacidad o el lugar no encajan, se descarta. No se inventan coordenadas.
- Si las coordenadas de la fuente contradicen el lugar que dice la propia ficha, no se usan.

In [ ]:
gem = pd.read_csv(GEM_CSV, sep=";", decimal=",", encoding="utf-8-sig")
col = gem[gem.country.eq("Colombia")].reset_index(drop=True)

PALABRAS = r"\b(agpe|gd|autog|planta|parque|solar|solares|fv|fotovoltaic[oa]|photovoltaic|power|plant|farm|project|pv|park|colombia|de|del|la|el|los|las|i|ii|iii|iv|v|vi)\b"
def normaliza(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(PALABRAS, " ", s)
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s).split())

col["nombre_norm"] = col.plant_name.map(normaliza)
xm = resumen[["code", "Name", "cap_MW"]].copy()

def candidatos(nombre, k=3):
    clave = normaliza(nombre)
    sim = col.nombre_norm.map(lambda t: difflib.SequenceMatcher(None, clave, t).ratio())
    return col.assign(similitud=sim).sort_values("similitud", ascending=False).head(k)

filas = []
for _, r in xm.iterrows():
    for _, c in candidatos(r.Name).iterrows():
        filas.append({"code": r.code, "xm": r.Name, "xm_MW": round(r.cap_MW, 1), "gem": c.plant_name,
                      "gem_MW": c.capacity, "gem_estado": c.operational_status, "similitud": round(c.similitud, 2)})
pd.DataFrame(filas).head(30)

In [ ]:
# Clave: código XM. Valor: (nombre exacto en GEM, motivo)
EMPAREJAMIENTOS = {
    "3DDT": ("La Loma Solar Park",             "150 MW en ambos. 'Latam Solar' es el desarrollador"),
    "3HF5": ("Fundacion (Latamsolar) solar farm", "100 MW en ambos. El parque Fundación de Enel está en Pivijay, Magdalena (pv magazine LATAM, 2023; Enel Green Power, 2023). "
                                                "El geocodificador de Open-Meteo ubica Pivijay en 10.462, -74.616, igual que esta fila de GEM. "
                                                "La otra fila, 'Fundación Solar Park' (10.518, -74.182), cae en el municipio de Fundación, 48 km al este, y se descarta"),
    "3INX": ("Caracolí Solar Plant",           "50 MW en ambos. GEM la marca pre-construcción pero XM la tiene generando"),
    "3J2B": ("Planeta Rica Solar Park",        "19.9 MW en ambos"),
    "4RGG": ("Baranoa (Cecropia) solar farm",  "19.9 MW en ambos. GEM la marca pre-construcción"),
    "5F43": ("Alma Solar PV solar farm",       "9.8 MW en ambos. Hay otra 'Alma solar farm' de 20 MW en otra ubicación. La ubicación de esta fila no se contrastó con una fuente externa y la planta tiene solo unos 60 días de datos"),
    "EPFV": ("El Paso PV Solar Park",          "68 MW en XM, 70 MW en GEM"),
    "GYPO": ("Guayepo PV Solar Park",          "370 MW en XM, 486.7 MW en GEM, que suma más etapas. Mismo sitio. XM registra Guayepo III como otro recurso de 200 MW"),
    "MATA": ("La Mata solar farm (Colombia)",  "80 MW en ambos"),
    "TPUY": ("Tepuy solar farm",               "83 MW en ambos"),
}

# Plantas que el dataset del paper no trae con ese nombre. Las coordenadas salen de la ficha pública de GEM,
# la misma fuente que alimenta el dataset. Cada una se contrastó con prensa o con el desarrollador.
GEM_WIKI = "https://www.gem.wiki/"
EXTRAS = {
    "3EFY": {"gem": "Los Llanos Solar Forest, fase 4", "lat": 4.2152, "lon": -72.0266, "precision": "aproximada (GEM)",
             "fuente": GEM_WIKI + "Los_Llanos_Solar_Forest",
             "motivo": "Puerto Gaitán, Meta. 19.9 MWac en GEM y en XM. Fases 4 y 5 inauguradas en julio de 2022 (pv magazine LATAM, 2022)"},
    "3EDL": {"gem": "Los Llanos Solar Forest, fase 5", "lat": 4.2108, "lon": -72.0310, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Los_Llanos_Solar_Forest",
             "motivo": "Puerto Gaitán, Meta. 17.9 MWac en GEM y en XM"},
    "3K8C": {"gem": "Bolívar PV Solar Forest, fase 503", "lat": 10.5732, "lon": -74.8904, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Bolivar_PV_Solar_Forest",
             "motivo": "Sabanalarga, Atlántico. BSB significa Bosques Solares de Bolívar, de ISAGEN, cinco plantas de 19.9 MW (pv magazine LATAM, 2024-07-29)"},
    "3K8E": {"gem": "Bolívar PV Solar Forest, fase 504", "lat": 10.5796, "lon": -74.8903, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Bolivar_PV_Solar_Forest",
             "motivo": "Sabanalarga, Atlántico. Está a unos 0.7 km de la fase 503. Que la fase 503 sea 3K8C y la 504 sea 3K8E se asume por la numeración, sin otra fuente"},
    "3IZ6": {"gem": "La Unión Solar Park (Colombia)", "lat": 8.6920, "lon": -75.7962, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "La_Uni%C3%B3n_Solar_Park",
             "motivo": "Montería, Córdoba. Solarpack (hoy Zelestra). 99.9 MWac en GEM y 100 MW en XM. El código XM y la fecha de arranque no se contrastaron con una fuente externa"},
    "3IRX": {"gem": "Portón Del Sol PV Solar Park", "lat": 5.4104, "lon": -74.7141, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Port%C3%B3n_Del_Sol_PV_Solar_Park",
             "motivo": "La Dorada, Caldas. Enerfín. 102 MW en GEM y en XM. El Espectador dice que entró en operación comercial el 8 de marzo de 2024, la misma fecha de la primera generación en XM"},
    "3IQA": {"gem": "Sunnorte Solar Plant", "lat": 8.2247, "lon": -73.3233, "precision": "exacta (GEM)",
             "fuente": GEM_WIKI + "Sunnorte_Solar_Plant",
             "motivo": "Ocaña, Norte de Santander. Ecoener. 35 MWac en GEM y en XM. GEM y Ecoener dicen 2023, y XM muestra la primera generación en noviembre de 2024. La diferencia de fechas no se explicó"},
}
NO_EMPAREJADAS = {
    "3PWF": "Autogenerador de 4.99 MW. Una hipótesis (Celsia Solar Palmira 2) no se confirmó con ninguna fuente. Queda fuera",
    "4YCF": "Shangri-La, 160 MW. Departamento confirmado (Tolima) pero las coordenadas de GEM caen unos 230 km al este de Ibagué y contradicen su propia ficha. Tiene además pocos días de datos",
}

sin_emparejar = xm[~xm.code.isin(list(EMPAREJAMIENTOS) + list(EXTRAS))]
print(f"Emparejadas con el dataset: {len(EMPAREJAMIENTOS)} | con la ficha de GEM: {len(EXTRAS)} | de {len(xm)}")
print("Sin emparejar (sin coordenadas, quedan fuera por ahora):")
print(sin_emparejar.assign(cap_MW=lambda d: d.cap_MW.round(1)).to_string(index=False))

In [ ]:
plantas = []
for code_xm, (nombre_gem, motivo) in EMPAREJAMIENTOS.items():
    g = col[col.plant_name == nombre_gem]
    assert len(g) == 1, f"{nombre_gem}: {len(g)} filas en GEM"
    g = g.iloc[0]
    plantas.append({"code": code_xm, "planta": xm.set_index("code").loc[code_xm, "Name"],
                    "lat": g.latitude, "lon": g.longitude, "cap_gem_MW": g.capacity, "estado_gem": g.operational_status})
for code_xm, e in EXTRAS.items():
    plantas.append({"code": code_xm, "planta": xm.set_index("code").loc[code_xm, "Name"],
                    "lat": e["lat"], "lon": e["lon"], "cap_gem_MW": np.nan, "estado_gem": "operating (ficha GEM)"})
plantas = pd.DataFrame(plantas)

# Grupos por cercanía: plantas a menos de CLUSTER_KM comparten clima y no son observaciones independientes
def haversine_km(a, b):
    la1, lo1, la2, lo2 = map(np.radians, [a[0], a[1], b[0], b[1]])
    h = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(h))

D = np.array([[haversine_km(a, b) for b in plantas[["lat", "lon"]].values] for a in plantas[["lat", "lon"]].values])
plantas["grupo"] = fcluster(linkage(squareform(D, checks=False), method="single"), CLUSTER_KM, criterion="distance")
print("Grupos de plantas a menos de", CLUSTER_KM, "km (enlace simple):", plantas.grupo.nunique())
plantas.round(3)

In [ ]:
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(EXTRAS).T[["gem", "lat", "lon", "precision", "fuente", "motivo"]])
print("Plantas de XM investigadas y descartadas:")
for k, v in NO_EMPAREJADAS.items():
    print(f"  {k}: {v}")

## 5. Clima por hora en cada planta

Open-Meteo devuelve por hora la radiación de onda corta (horizontal, promedio de la hora anterior), la nubosidad total, la temperatura,
la humedad y el viento. Pedimos también la radiación en plano inclinado (`global_tilted_irradiance`) en dos variantes: inclinación fija de 10° orientada
al sur, y el modo `tilt=nan` que Open-Meteo documenta como seguidor horizontal. No verificamos que ese modo equivalga a un seguidor de un eje norte-sur que sigue al sol de este a oeste, así que su resultado no se interpreta. Los 10° son una suposición nuestra para un sitio cercano al ecuador. La tecnología
real de cada planta no la conocemos.

Se piden dos productos para las mismas coordenadas: `era5` y `best_match`. Open-Meteo guarda cada valor en la celda de su modelo, y la celda más cercana a una
planta cambia con el modelo. Esa diferencia sirve para ver cuánto depende el resultado de la fuente de radiación.

In [ ]:
BASE_VARS = "shortwave_radiation,cloud_cover,temperature_2m,relative_humidity_2m,wind_speed_10m"

def om_get(params, ruta):
    if ruta.exists():
        return pd.read_csv(ruta)
    ultimo_error = None
    for intento in range(6):
        try:
            r = requests.get(OM_URL, params=params, timeout=180)
            h = pd.DataFrame(r.json()["hourly"])
            break
        except (requests.RequestException, ValueError, KeyError) as e:
            ultimo_error = e
            time.sleep(10 * (intento + 1))      # Open-Meteo limita la tasa de consultas; esperamos y reintentamos
    else:
        raise RuntimeError(f"Open-Meteo no devolvió datos ({ruta.name}): {ultimo_error}")
    h.to_csv(ruta, index=False)
    time.sleep(5)
    return h

def clima_planta(code_xm, lat, lon):
    comunes = {"latitude": lat, "longitude": lon, "start_date": str(START), "end_date": str(END), "timezone": "America/Bogota"}
    out = {}
    for modelo in ["era5", "best_match"]:
        out[modelo] = om_get({**comunes, "hourly": BASE_VARS, "models": modelo}, DATA / f"clima_{code_xm}_{modelo}.csv")
    out["gti10"] = om_get({**comunes, "hourly": "global_tilted_irradiance", "tilt": 10, "azimuth": 0}, DATA / f"clima_{code_xm}_gti10.csv")
    out["gtitrk"] = om_get({**comunes, "hourly": "global_tilted_irradiance", "tilt": "nan", "azimuth": 0}, DATA / f"clima_{code_xm}_gtitrk.csv")
    return out

crudo = {p.code: clima_planta(p.code, p.lat, p.lon) for p in plantas.itertuples()}
# Celda del modelo que respondió para cada planta (se guarda para no repetir la consulta)
ruta_celdas = DATA / "celdas_modelo.json"
celdas = json.loads(ruta_celdas.read_text()) if ruta_celdas.exists() else {}
for p in plantas.itertuples():
    for modelo in ["era5", "best_match"]:
        clave = f"{p.code}_{modelo}"
        if clave in celdas:
            continue
        try:
            r = requests.get(OM_URL, params={"latitude": p.lat, "longitude": p.lon, "start_date": "2025-03-01", "end_date": "2025-03-01",
                                             "hourly": "shortwave_radiation", "models": modelo}, timeout=60).json()
            celdas[clave] = [r["latitude"], r["longitude"]]
        except (requests.RequestException, ValueError, KeyError):
            continue
        time.sleep(1)
ruta_celdas.write_text(json.dumps(celdas))

filas = []
for p in plantas.itertuples():
    f = {"planta": p.planta, "lat": round(p.lat, 3), "lon": round(p.lon, 3)}
    for modelo in ["era5", "best_match"]:
        c = celdas.get(f"{p.code}_{modelo}")
        f[f"km_a_celda_{modelo}"] = round(haversine_km((p.lat, p.lon), c), 1) if c else np.nan
    filas.append(f)
pd.DataFrame(filas)

### Resúmenes diarios

Para evitar problemas de alineación entre las horas de XM y las de Open-Meteo trabajamos con resúmenes diarios.
Un revisor probó desfases de -2 a +2 horas con datos horarios y la correlación fue máxima sin desfase, así que las etiquetas coinciden sin corrección.

- Radiación diaria en kWh/m²: suma horaria dividida entre 1000.
- Nubosidad media en horas de luz (radiación mayor que 0).
- Temperatura, humedad y viento: medias diarias.
- Índice físico de energía: radiación horaria multiplicada por un factor de temperatura. La temperatura de celda sigue la regla de NOCT,
  `Tc = Ta + (NOCT - 20) / 800 · G`, con NOCT de 45 °C (Skoplaki y Palyvos, 2009, *Renewable Energy*). La pérdida es de 0.4 % por grado sobre 25 °C, un valor típico
  dentro del rango que revisan Skoplaki y Palyvos (2009, *Solar Energy*). Son supuestos genéricos, no los datos de cada planta.

In [ ]:
NOCT, GAMMA = 45.0, -0.004

def energia_fisica(g, ta):
    tc = ta + (NOCT - 20.0) / 800.0 * g
    return (g * (1 + GAMMA * (tc - 25.0))).clip(lower=0)

def resume(h, sufijo, extra_clima=True):
    h = h.copy()
    h["date"] = pd.to_datetime(h.time.str[:10])
    g = h.shortwave_radiation
    h["e_fis"] = energia_fisica(g, h.temperature_2m)
    luz = h.shortwave_radiation > 0
    out = h.groupby("date").agg(rad=("shortwave_radiation", lambda s: s.sum() / 1000),
                                e_fis=("e_fis", lambda s: s.sum() / 1000))
    out["nube"] = h[luz].groupby("date").cloud_cover.mean()
    if extra_clima:
        out["temp"] = h.groupby("date").temperature_2m.mean()
        out["humedad"] = h.groupby("date").relative_humidity_2m.mean()
        out["viento"] = h.groupby("date").wind_speed_10m.mean()
    return out.add_suffix(sufijo).reset_index()

def gti_diaria(h, nombre):
    h = h.copy(); h["date"] = pd.to_datetime(h.time.str[:10])
    return h.groupby("date").global_tilted_irradiance.sum().div(1000).rename(nombre).reset_index()

clima = []
for p in plantas.itertuples():
    c = crudo[p.code]
    t = (resume(c["best_match"], "_bm")
         .merge(resume(c["era5"], "_e5", extra_clima=False), on="date")
         .merge(gti_diaria(c["gti10"], "gti10"), on="date")
         .merge(gti_diaria(c["gtitrk"], "gtitrk"), on="date"))
    clima.append(t.assign(code=p.code))
clima = pd.concat(clima, ignore_index=True)
clima.describe().round(2).T[["count", "mean", "std", "min", "max"]]

## 6. Tabla diaria final

Una fila por planta y día con el FC real y el clima de ese día. Se guarda en `tabla_diaria_colombia.csv` para que otros cuadernos partan de ahí.
La tabla principal excluye los primeros `RAMP_DAYS` días de las plantas con arranque observado y las plantas que quedan con menos de `MIN_DIAS` días útiles. Las tablas sin esos filtros se conservan para la sensibilidad de la sección 9.4.

In [ ]:
completa = (diario.merge(plantas[["code", "planta", "lat", "lon", "grupo"]], on="code")
                  .merge(clima, on=["code", "date"], how="inner")
                  .dropna(subset=["rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm", "gti10", "gtitrk"])
                  .sort_values(["code", "date"]).reset_index(drop=True))
completa["fc_ayer"] = completa.groupby("code").fc.shift(1)
completa.loc[completa.date.diff().dt.days.ne(1) | completa.code.ne(completa.code.shift(1)), "fc_ayer"] = np.nan

tabla_base = completa[completa.dias_desde_arranque >= RAMP_DAYS].reset_index(drop=True)
dias_utiles = tabla_base.groupby("code").size()
validas = dias_utiles[dias_utiles >= MIN_DIAS].index
print("Plantas con menos de", MIN_DIAS, "días útiles tras excluir el arranque:", dias_utiles[dias_utiles < MIN_DIAS].to_dict())
tabla = tabla_base[tabla_base.code.isin(validas)].reset_index(drop=True)
tabla.to_csv("tabla_diaria_colombia.csv", index=False)

print(f"Filas (principal): {len(tabla)} | plantas: {tabla.code.nunique()} | grupos de cercanía: {tabla.grupo.nunique()}")
print(f"Filas excluidas por puesta en marcha: {len(completa) - len(tabla_base)}")
print(tabla.groupby("planta").agg(dias=("fc", "size"), fc_medio=("fc", "mean")).round(3).to_string())

## 6b. EDA de la base de datos

### Diccionario de variables de `tabla_diaria_colombia.csv`

Una fila es una planta en un día.

| Columna | Qué es | Unidad | Fuente |
|---|---|---|---|
| `code` | Código del recurso en XM | texto | XM |
| `planta` | Nombre del recurso en XM | texto | XM |
| `date` | Día, hora local de Bogotá | fecha | XM |
| `kwh` | Energía generada en el día (suma de las 24 horas) | kWh | XM, métrica `Gene` |
| `cap_MW` | Capacidad efectiva neta de ese día | MW | XM, métrica `CapEfecNeta` |
| `fc` | Factor de capacidad: `kwh / (cap_MW · 1000 · 24)`. Es la variable que se predice | fracción de 0 a 1 | calculada |
| `fc_ayer` | `fc` del día anterior, vacío si el día anterior falta | fracción | calculada |
| `primer_dia` | Primer día con generación y capacidad en la ventana | fecha | calculada |
| `dias_desde_arranque` | Días desde `primer_dia`. Vale 10000 si la planta ya operaba al inicio de la ventana | días | calculada |
| `lat`, `lon` | Coordenadas de la planta | grados | dataset del paper o ficha de GEM (sección 4) |
| `grupo` | Zona de plantas a menos de 100 km entre sí | entero | calculada |
| `rad_bm` | Radiación global horizontal del día, modelo `best_match` | kWh/m²/día | Open-Meteo |
| `rad_e5` | Lo mismo con el modelo `era5` | kWh/m²/día | Open-Meteo |
| `e_fis_bm`, `e_fis_e5` | Radiación horaria con la pérdida por temperatura de celda (NOCT 45 °C, 0.4 % por °C), sumada en el día | kWh/m²/día | calculada |
| `nube_bm`, `nube_e5` | Nubosidad total media en las horas de luz | % | Open-Meteo |
| `temp_bm` | Temperatura media del aire a 2 m | °C | Open-Meteo |
| `humedad_bm` | Humedad relativa media a 2 m | % | Open-Meteo |
| `viento_bm` | Velocidad media del viento a 10 m | km/h | Open-Meteo |
| `gti10` | Radiación en un plano fijo de 10° orientado al sur | kWh/m²/día | Open-Meteo |
| `gtitrk` | Radiación con el modo `tilt=nan` de Open-Meteo (no interpretado, ver sección 5) | kWh/m²/día | Open-Meteo |

La tecnología de cada planta (seguidor o estructura fija, potencia pico DC frente a capacidad AC) no está en la base. Queda pendiente: es lo que más
probablemente explica por qué unas plantas tienen un nivel de FC más alto que otras.

In [ ]:
print("Dimensiones:", tabla.shape, "| plantas:", tabla.code.nunique(), "| zonas:", tabla.grupo.nunique(),
      "| del", tabla.date.min().date(), "al", tabla.date.max().date())
VARS_EDA = ["fc", "kwh", "cap_MW", "rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
tabla[VARS_EDA].describe().T.round(3)

### Valores faltantes

Para cada planta se arma la rejilla de todos los días entre su primer día con datos y el 28 de febrero de 2026, y se cuenta qué falta en cada día.
Las horas vacías dentro de un día en XM son horas de noche sin generación, así que no cuentan como faltantes. Un día cuenta como faltante en generación
si XM no trae registro o trae 0 kWh.

In [ ]:
plantas_ok = tabla.code.unique()
filas = []
for code in plantas_ok:
    ini = diario.loc[diario.code == code, "date"].min()
    for d in pd.date_range(ini, pd.Timestamp(END)):
        filas.append((code, d))
rejilla = pd.DataFrame(filas, columns=["code", "date"])
g_dia = gen.groupby(["code", "date"]).kwh.sum().reset_index()
rejilla = (rejilla.merge(g_dia, on=["code", "date"], how="left")
                  .merge(cap_dia, on=["code", "date"], how="left")
                  .merge(clima[["code", "date", "rad_bm", "rad_e5", "nube_bm"]], on=["code", "date"], how="left"))

falt = pd.DataFrame({
    "dias_esperados": rejilla.groupby("code").size(),
    "sin_generacion": rejilla.assign(x=rejilla.kwh.isna() | (rejilla.kwh <= 0)).groupby("code").x.sum(),
    "sin_capacidad": rejilla.assign(x=rejilla.cap_MW.isna() | (rejilla.cap_MW <= 0)).groupby("code").x.sum(),
    "sin_clima": rejilla.assign(x=rejilla.rad_bm.isna() | rejilla.rad_e5.isna() | rejilla.nube_bm.isna()).groupby("code").x.sum(),
    "dias_en_tabla": tabla.groupby("code").size(),
}).join(tabla.groupby("code").planta.first())
falt["pct_faltante"] = 100 * (1 - (falt.dias_en_tabla + 0) / falt.dias_esperados)
print("Faltantes por columna en la tabla final:", int(tabla.isna().sum().drop("fc_ayer").sum()), "valores vacíos fuera de fc_ayer")
print("fc_ayer vacío:", int(tabla.fc_ayer.isna().sum()), "filas (primer día de cada planta o día anterior ausente)")
falt.set_index("planta").sort_values("pct_faltante", ascending=False).round(1)

La columna `pct_faltante` mezcla dos causas: días que faltan en XM y los días de puesta en marcha que se excluyen a propósito. Las plantas con arranque observado
pierden sus primeros 60 días por diseño, y eso no es un dato faltante.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), gridspec_kw={"width_ratios": [1.3, 1.2, 1]})

orden = tabla.groupby("planta").fc.median().sort_values().index
axes[0].boxplot([tabla.loc[tabla.planta == p, "fc"] for p in orden], vert=False, showfliers=False)
axes[0].set_yticks(range(1, len(orden) + 1)); axes[0].set_yticklabels([p[:24] for p in orden], fontsize=8)
axes[0].set_xlabel("factor de capacidad diario"); axes[0].set_title("Distribución del FC por planta")

cob = (tabla.assign(mes=tabla.date.dt.to_period("M").astype(str)).groupby(["planta", "mes"]).size().unstack(fill_value=0))
im = axes[1].imshow(cob.values, aspect="auto", cmap="Greens", vmin=0, vmax=31)
axes[1].set_yticks(range(len(cob.index))); axes[1].set_yticklabels([p[:24] for p in cob.index], fontsize=8)
axes[1].set_xticks(range(0, len(cob.columns), 3)); axes[1].set_xticklabels(cob.columns[::3], rotation=90, fontsize=8)
axes[1].set_title("Días con datos por planta y mes"); fig.colorbar(im, ax=axes[1], shrink=.7)

VARS_CORR = ["fc", "rad_bm", "rad_e5", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
cen = tabla[VARS_CORR + ["code"]].copy()
for v in VARS_CORR:
    cen[v] = cen[v] - cen.groupby("code")[v].transform("mean")
cor = cen[VARS_CORR].corr(method="spearman")
im2 = axes[2].imshow(cor.values, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_xticks(range(len(VARS_CORR))); axes[2].set_xticklabels(VARS_CORR, rotation=90)
axes[2].set_yticks(range(len(VARS_CORR))); axes[2].set_yticklabels(VARS_CORR)
for i in range(len(VARS_CORR)):
    for j in range(len(VARS_CORR)):
        axes[2].text(j, i, f"{cor.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
axes[2].set_title("Spearman dentro de planta (variables centradas)")
fig.tight_layout(); fig.savefig("fig_eda_base_colombia.png", dpi=130, bbox_inches="tight"); plt.show()

La matriz de la derecha usa las variables con la media de cada planta restada, así que mide cómo se mueven juntas de un día a otro. La radiación
y la nubosidad están muy correlacionadas, igual que la temperatura y la humedad con la radiación. Eso explica por qué añadir esas variables a un modelo
que ya tiene la radiación aporta poco.

In [ ]:
meta = plantas[plantas.code.isin(tabla.code.unique())][["code", "planta", "lat", "lon", "grupo"]].copy()
meta["fuente_coordenadas"] = meta.code.map(lambda c: "dataset del paper: " + EMPAREJAMIENTOS[c][0] if c in EMPAREJAMIENTOS else "ficha GEM: " + EXTRAS[c]["fuente"])
meta["precision"] = meta.code.map(lambda c: EXTRAS[c]["precision"] if c in EXTRAS else "la del dataset (GEM)")
meta["motivo"] = meta.code.map(lambda c: EMPAREJAMIENTOS[c][1] if c in EMPAREJAMIENTOS else EXTRAS[c]["motivo"])
# Lo poco que se sabe de tecnología, con su fuente. Todo lo demás queda pendiente.
DC_CONOCIDO = {
    "3EFY": ("27.39", "https://www.pv-magazine-latam.com/2022/07/15/inauguran-en-colombia-las-fases-4-y-5-de-bosques-de-los-llanos-complejo-solar-que-completa-125-mwp/"),
    "3EDL": ("25.01", "https://www.pv-magazine-latam.com/2022/07/15/inauguran-en-colombia-las-fases-4-y-5-de-bosques-de-los-llanos-complejo-solar-que-completa-125-mwp/"),
    "3IQA": ("41", "https://www.ecoener.es/proyectos/planta-fotovoltaica-sunnorte"),
}
meta["dc_MWp"] = meta.code.map(lambda c: DC_CONOCIDO.get(c, ("pendiente", ""))[0])
meta["fuente_dc"] = meta.code.map(lambda c: DC_CONOCIDO.get(c, ("", ""))[1])
meta["montaje"] = "pendiente"
meta = meta.merge(tabla.groupby("code").agg(dias=("fc", "size"), fc_medio=("fc", "mean"), cap_MW=("cap_MW", "median")).reset_index(), on="code")
meta.to_csv("plantas_colombia.csv", index=False)
meta[["code", "planta", "lat", "lon", "grupo", "dias", "fc_medio", "cap_MW", "precision"]].round(3)

### Tecnología de cada planta: pendiente

La tabla siguiente muestra lo que falta. Ninguna planta tiene confirmado el tipo de montaje (seguidor de un eje o estructura fija). Solo tres tienen una potencia
pico DC citada por una fuente, que reportó un investigador y no se revisó a mano. Mientras esto no se complete, la diferencia de nivel entre plantas
(el 16 % de la varianza del FC) queda sin explicar.

In [ ]:
tec = meta[["code", "planta", "cap_MW", "dc_MWp", "montaje", "fuente_dc"]].copy()
tec["dc_ac"] = pd.to_numeric(tec.dc_MWp, errors="coerce") / tec.cap_MW
print("Plantas sin montaje confirmado:", int((tec.montaje == "pendiente").sum()), "de", len(tec))
print("Plantas sin potencia DC:", int((tec.dc_MWp == "pendiente").sum()), "de", len(tec))
tec.round(2)

## 7. Qué se ve antes de modelar

Una gráfica por planta del FC contra la radiación del día, coloreada por nubosidad. Si el clima explica la generación, los puntos suben con la radiación.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
for ax, (nombre, g) in zip(axes.ravel(), tabla.groupby("planta")):
    sc = ax.scatter(g.rad_bm, g.fc, c=g.nube_bm, s=6, cmap="viridis_r", alpha=.7)
    ax.set_title(f"{nombre[:22]}  rho={spearmanr(g.rad_bm, g.fc)[0]:.2f}", fontsize=9)
    ax.set_xlabel("radiación (kWh/m²/día)")
axes[0, 0].set_ylabel("factor de capacidad"); axes[1, 0].set_ylabel("factor de capacidad")
fig.colorbar(sc, ax=axes, label="nubosidad media de día (%)", shrink=.8)
fig.suptitle("Factor de capacidad real contra radiación modelada (best_match), por planta", y=.98)
fig.savefig("fig_fc_vs_radiacion.png", dpi=130, bbox_inches="tight")
plt.show()

cols = ["rad_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"]
(tabla.groupby("planta").apply(lambda g: pd.Series({v: spearmanr(g[v], g.fc)[0] for v in cols})).round(2))

La temperatura sale con correlación positiva con el FC en todas las plantas. Físicamente el calor reduce el rendimiento, así que esa
correlación viene de que los días soleados también son más cálidos. No se debe leer como un efecto de la temperatura.

## 8. Cómo se evalúa

Hay tres decisiones de diseño que vienen de la revisión metodológica de este cuaderno.

Nivel de cada planta. Cada planta tiene su propio nivel de FC por factores que no medimos (seguidores, relación DC/AC, edad). La varianza entre plantas se calcula al empezar la sección 9.1.
Para medir solo lo que el clima puede explicar, a cada variable se le resta la media de su planta y se predice el sube y baja diario. Eso presupone conocer el nivel de la planta.
En la evaluación temporal el nivel se calcula solo con la primera mitad de los días de cada planta.

Plantas que no se pueden tratar como independientes. Varias plantas están a menos de 100 km y comparten clima el mismo día. La validación deja fuera un grupo
completo de plantas cercanas a la vez (`LeaveOneGroupOut` sobre `grupo`), no una planta suelta.

Incertidumbre. Con 16 plantas en 6 zonas, la unidad de muestreo es la planta. Los intervalos salen de remuestrear plantas (Field y Welsh, 2007), no filas.
Las comparaciones entre modelos usan diferencias pareadas por planta con la prueba de Wilcoxon. Con 16 pares la prueba tiene poca potencia, así que todo es exploratorio.

Métricas. R² dentro de la planta, error absoluto medio relativo al FC medio de esa planta (MAE relativo) y la mejora frente a dos referencias sin clima:
predecir siempre el nivel de la planta y repetir el FC del día anterior.

In [ ]:
FEATS = {
    "A. radiación era5":                        ["rad_e5"],
    "B. radiación best_match":                  ["rad_bm"],
    "C. física: radiación con temperatura":     ["e_fis_bm"],
    "D. plano con tilt=nan (modo seguidor de Open-Meteo)": ["gtitrk"],
    "E. radiación en plano fijo de 10°":        ["gti10"],
    "F. radiación + nubosidad":                 ["rad_bm", "nube_bm"],
    "G. todas (rad, nube, temp, humedad, viento)": ["rad_bm", "nube_bm", "temp_bm", "humedad_bm", "viento_bm"],
    "H. solo nubosidad":                        ["nube_bm"],
}

def centrar(df, cols):
    c = df.copy()
    for v in cols:
        c[v + "_c"] = c[v] - c.groupby("code")[v].transform("mean")
    return c

def metricas_planta(y_real, y_pred, y_media, y_ayer):
    sst = ((y_real - y_real.mean()) ** 2).sum()
    mae = np.abs(y_real - y_pred).mean()
    mae_nivel = np.abs(y_real - y_media).mean()
    ok = ~np.isnan(y_ayer)
    mae_pers = np.abs(y_real[ok] - y_ayer[ok]).mean() if ok.sum() > 5 else np.nan
    return {"r2": 1 - ((y_real - y_pred) ** 2).sum() / sst, "mae_rel": mae / y_media,
            "mejora_vs_nivel": 1 - mae / mae_nivel,
            "mejora_vs_persistencia": 1 - np.abs(y_real[ok] - y_pred[ok]).mean() / mae_pers if ok.sum() > 5 else np.nan}

def evalua_nivel_conocido(df, feats, modelo=LinearRegression):
    cols = feats + ["fc"]
    c = centrar(df, cols)
    filas = []
    for tr, te in LeaveOneGroupOut().split(c, groups=c.grupo):
        m = modelo().fit(c.iloc[tr][[v + "_c" for v in feats]], c.iloc[tr].fc_c)
        d = c.iloc[te].copy()
        d["pred_c"] = m.predict(d[[v + "_c" for v in feats]])
        for code, g in d.groupby("code"):
            media = df[df.code == code].fc.mean()
            y = g.fc.values; pred = g.pred_c.values + media
            filas.append({"code": code, "planta": g.planta.iloc[0], "n": len(g),
                          **metricas_planta(y, pred, media, g.fc_ayer.values)})
    return pd.DataFrame(filas)

def evalua_temporal(df, feats, modelo=LinearRegression):
    d = df.copy()
    d["mitad"] = d.groupby("code").cumcount() >= d.groupby("code").fc.transform("size") // 2
    ent, prueba = d[~d.mitad], d[d.mitad]
    medias = ent.groupby("code")[feats + ["fc"]].mean()
    ent_c = ent.copy(); prueba_c = prueba.copy()
    for v in feats + ["fc"]:
        ent_c[v + "_c"] = ent[v] - ent.code.map(medias[v]); prueba_c[v + "_c"] = prueba[v] - prueba.code.map(medias[v])
    filas = []
    for g_id in d.grupo.unique():
        tr, te = ent_c[ent_c.grupo != g_id], prueba_c[prueba_c.grupo == g_id]
        m = modelo().fit(tr[[v + "_c" for v in feats]], tr.fc_c)
        for code, g in te.groupby("code"):
            media = medias.loc[code, "fc"]
            pred = m.predict(g[[v + "_c" for v in feats]]) + media
            filas.append({"code": code, "planta": g.planta.iloc[0], "n": len(g), **metricas_planta(g.fc.values, pred, media, g.fc_ayer.values)})
    return pd.DataFrame(filas)

def ic_bootstrap(valores, B=5000):
    v = np.asarray(valores); n = len(v)
    med = [np.median(v[RNG.integers(0, n, n)]) for _ in range(B)]
    return np.percentile(med, [2.5, 97.5])

def tabla_resumen(funcion, conjuntos=FEATS, df=None):
    df = tabla if df is None else df
    filas, por_planta = [], {}
    for nombre, feats in conjuntos.items():
        r = funcion(df, feats)
        por_planta[nombre] = r.set_index("code")
        lo, hi = ic_bootstrap(r.r2)
        filas.append({"variables": nombre, "r2_mediana": r.r2.median(), "ic95_bajo": lo, "ic95_alto": hi,
                      "r2_min": r.r2.min(), "r2_max": r.r2.max(), "mae_rel_medio": r.mae_rel.mean(),
                      "mejora_vs_nivel": r.mejora_vs_nivel.mean(), "mejora_vs_persistencia": r.mejora_vs_persistencia.mean()})
    return pd.DataFrame(filas).set_index("variables"), por_planta

## 9. Resultados

### 9.1 Nivel de la planta conocido

El nivel de cada planta se calcula con todos sus días, y la validación deja fuera un grupo completo de plantas cercanas.

In [ ]:
w = tabla.groupby("code").fc.transform(lambda s: s - s.mean()).var()
print(f"Varianza del FC entre plantas: {100 * (1 - w / tabla.fc.var()):.0f} %  |  dentro de cada planta, día a día: {100 * w / tabla.fc.var():.0f} %")

COLS_1 = ["r2_mediana", "ic95_bajo", "ic95_alto", "r2_min", "r2_max"]
COLS_2 = ["mae_rel_medio", "mejora_vs_nivel", "mejora_vs_persistencia"]
res_A, pp_A = tabla_resumen(evalua_nivel_conocido)
display(res_A[COLS_1].round(3)); display(res_A[COLS_2].round(3))

In [ ]:
c = centrar(tabla, ["rad_bm", "fc"])
pend = np.polyfit(c.rad_bm_c, c.fc_c, 1)[0]
print(f"Pendiente dentro de planta: {100 * pend:.1f} puntos de FC por cada kWh/m² de radiación diaria")
print(f"Proporcionalidad simple: FC medio / radiación media = {100 * tabla.fc.mean() / tabla.rad_bm.mean():.1f} puntos por kWh/m²")
print(f"Plantas: {tabla.code.nunique()} | grupos de cercanía: {tabla.grupo.nunique()} | filas: {len(tabla)} | del {tabla.date.min().date()} al {tabla.date.max().date()}")

In [ ]:
base = "B. radiación best_match"
filas = []
for nombre, r in pp_A.items():
    if nombre == base:
        continue
    d = (r.r2 - pp_A[base].r2).dropna()
    p = wilcoxon(d).pvalue if (d != 0).any() else np.nan
    filas.append({"modelo": nombre, "dif_r2_media_vs_B": d.mean(), "dif_min": d.min(), "dif_max": d.max(), "p_wilcoxon": p})
pd.DataFrame(filas).set_index("modelo").round(3)

Los valores p son crudos, de una prueba con 16 pares y 7 comparaciones. No se corrigen por comparaciones múltiples porque no hay una hipótesis confirmatoria: se leen como orden de magnitud de la diferencia.

### 9.2 Nivel estimado con el pasado

Aquí el nivel de cada planta sale solo de la primera mitad de sus días, y se evalúa en la segunda mitad. Es una prueba más dura: las plantas cuyo nivel se desplaza en el tiempo salen peor.

In [ ]:
res_B, pp_B = tabla_resumen(evalua_temporal)
display(res_B[COLS_1].round(3)); display(res_B[COLS_2].round(3))

In [ ]:
pd.DataFrame({"r2 nivel conocido (B)": pp_A[base].r2, "r2 temporal (B)": pp_B[base].r2,
              "r2 nivel conocido (C física)": pp_A["C. física: radiación con temperatura"].r2,
              "r2 temporal (C física)": pp_B["C. física: radiación con temperatura"].r2}
             ).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

### 9.3 Un modelo flexible

Para comprobar si un modelo no lineal cambia algo, se prueba HistGradientBoosting (Friedman, 2001) con todas las variables centradas, con la misma validación por grupos.
Sin hyperparameter tuning y con 16 plantas, una pérdida frente al modelo lineal puede venir del sobreajuste del propio modelo, no del clima.

In [ ]:
r_hgb = evalua_nivel_conocido(tabla, FEATS["G. todas (rad, nube, temp, humedad, viento)"],
                              modelo=lambda: HistGradientBoostingRegressor(random_state=SEED))
lo, hi = ic_bootstrap(r_hgb.r2)
print(f"HGB, todas las variables centradas: R² mediana {r_hgb.r2.median():.3f}  (IC95 {lo:.3f} a {hi:.3f})")
print(f"Lineal, mismas variables:           R² mediana {res_A.loc['G. todas (rad, nube, temp, humedad, viento)', 'r2_mediana']:.3f}")

### 9.4 Sensibilidad a decisiones previas

Cada fila cambia una decisión y repite la evaluación con solo radiación (B) y con la física (C).

In [ ]:
def sens(df, etiqueta):
    cj = {k: FEATS[k] for k in ["B. radiación best_match", "C. física: radiación con temperatura"]}
    r, _ = tabla_resumen(evalua_nivel_conocido, cj, df)
    return r[["r2_mediana", "ic95_bajo", "ic95_alto"]].assign(escenario=etiqueta, n_filas=len(df)).reset_index()

esc = [
    sens(tabla, "principal (60 días de arranque excluidos)"),
    sens(completa[completa.code.isin(validas)], "sin excluir días de arranque"),
    sens(completa[completa.code.isin(validas) & (completa.dias_desde_arranque >= 90)], "90 días de arranque excluidos"),
    sens(tabla_base, "incluyendo ALMA II (8 días útiles)"),
    sens(tabla[tabla.fc >= 0.02], "sin días con FC < 0.02 (filtra por la respuesta, solo sensibilidad)"),
]
pd.concat(esc).set_index(["escenario", "variables"]).round(3)

### 9.5 Cuánto pesa la fuente de radiación

Los modelos A y B usan la misma regresión sobre la radiación de dos productos de Open-Meteo. Si la diferencia entre ellos es grande, la calidad del modelo de clima
es parte del techo del resultado.

In [ ]:
pd.DataFrame({"era5 (0.25°)": pp_A["A. radiación era5"].r2, "best_match": pp_A["B. radiación best_match"].r2}
            ).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

### 9.6 Modelos del curso, versión preliminar

Prueba de los modelos base que pide el curso sobre esta base: regresión lineal, Ridge, Lasso, KNN y SVR con kernel RBF, más HistGradientBoosting
como modelo flexible y la referencia de predecir el nivel de la planta.

Cómo se evita la fuga de información:
- La partición va primero. La validación deja fuera una zona completa de plantas cercanas (`LeaveOneGroupOut` sobre `grupo`).
- El nivel de cada planta se calcula solo con la primera mitad de sus días (el pasado) y se evalúa en la segunda mitad. Las plantas de la zona de prueba no entran al entrenamiento.
- El escalado (`StandardScaler`) va dentro de un `Pipeline`, así que se ajusta solo con los datos de entrenamiento de cada partición.
- Los hiperparámetros quedan fijos en valores por defecto razonables. No se ajustan aquí, porque ajustarlos con los mismos datos de prueba inflaría el resultado. El ajuste va en la versión final, con validación anidada por grupos.

Variables: radiación, nubosidad, temperatura, humedad y viento (`best_match`).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

class NivelPlanta:
    # Predice desviación 0: el FC es el nivel de la planta calculado con el pasado
    def fit(self, X, y): return self
    def predict(self, X): return np.zeros(len(X))

MODELOS_CURSO = {
    "nivel de la planta (referencia)": lambda: NivelPlanta(),
    "regresión lineal": lambda: make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha=1)": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (alpha=0.001)": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.001, max_iter=10000)),
    "KNN (k=25)": lambda: make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=25)),
    "SVR RBF (C=1, epsilon=0.01)": lambda: make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.01)),
    "HistGradientBoosting": lambda: HistGradientBoostingRegressor(random_state=SEED),
}
VARS_G = FEATS["G. todas (rad, nube, temp, humedad, viento)"]
filas, por_planta_curso = [], {}
for nombre, fab in MODELOS_CURSO.items():
    t0 = time.time()
    r = evalua_temporal(tabla, VARS_G, modelo=fab)
    por_planta_curso[nombre] = r.set_index("code").r2
    lo, hi = ic_bootstrap(r.r2)
    filas.append({"modelo": nombre, "r2_mediana": r.r2.median(), "ic95_bajo": lo, "ic95_alto": hi,
                  "mae_rel_medio": r.mae_rel.mean(), "mejora_vs_persistencia": r.mejora_vs_persistencia.mean(),
                  "segundos": time.time() - t0})
res_curso = pd.DataFrame(filas).set_index("modelo")
res_curso.round(3)

In [ ]:
pd.DataFrame(por_planta_curso).join(tabla.groupby("code").planta.first()).set_index("planta").round(2)

## 10. Modelos propuestos para la versión final

| Modelo | Por qué |
|---|---|
| Referencia física (radiación por un factor propio de cada planta) | Es lo que predice la física. Cualquier modelo de aprendizaje debe superarla para justificar su complejidad |
| Ridge y Lasso | Lineales, interpretables y con regularización. La relación con la radiación es casi lineal (sección 9.1), y Lasso muestra qué variables sobran |
| KNN y SVR con kernel | Modelos base del curso. Capturan no linealidades sin suponer una forma, aunque con pocas zonas pueden sobreajustar |
| HistGradientBoosting o XGBoost | Interacciones entre variables. En la prueba preliminar no superaron al lineal, así que entran como control |

Diseño para la versión final: los mismos grupos de zonas, el nivel de cada planta estimado con el pasado, y la búsqueda de hiperparámetros dentro de un
bucle interno que también agrupe por zona (validación anidada), con el escalado dentro del `Pipeline`. Lo que más cambiaría los resultados no es el
modelo sino los datos: la tecnología de cada planta, datos horarios y radiación medida.

## 11. Conclusiones

Muestra: 16 plantas con coordenadas verificadas y al menos 100 días útiles, en 6 zonas de cercanía, del 1 de enero de 2024 al 28 de febrero de 2026, con 8,589 filas planta-día y sin valores vacíos en la tabla final. Nueve plantas salen del dataset del paper y siete de la ficha pública de GEM, cada una contrastada con prensa o con el desarrollador. ALMA II sale por quedar con 8 días útiles, Palmira II porque su ubicación no se confirmó, y Shangri-La porque sus coordenadas en GEM contradicen su propia ficha.

Lo que muestran los resultados con el nivel de cada planta conocido (sección 9.1):
- La radiación diaria de Open-Meteo (`best_match`) explica una mediana de R² de 0.49 de la variación día a día dentro de una planta (IC95 de 0.38 a 0.51 remuestreando plantas). Por planta va de 0.09 (Sunnorte, en Ocaña) a 0.59 (Caracolí). Esa variación diaria es el 84 % de la varianza total del FC.
- Cada kWh/m² adicional de radiación diaria suma unos 4.7 puntos de factor de capacidad, casi lo mismo que sale de dividir el FC medio entre la radiación media (4.6). El resultado es el que predice la física, y lo que aporta este cuaderno es medirlo en 16 plantas colombianas reales.
- Frente a predecir el nivel de la planta, el error absoluto baja un 28 %. Frente a repetir el FC de ayer, baja un 31 %.
- La pérdida por temperatura de celda (0.4 % por °C, NOCT de 45 °C) no mejora la radiación sola (0.487 frente a 0.493). Los parámetros son genéricos.
- Añadir nubosidad, temperatura, humedad y viento sube el R² por planta en promedio 0.018 (de -0.012 a +0.036). Es un efecto pequeño. La nubosidad sola explica 0.17.
- La fuente de radiación pesa: con `era5` la mediana es 0.41 y con `best_match` 0.49. `best_match` gana en 13 de las 16 plantas.

Modelos del curso con el nivel calculado con el pasado y el escalado dentro de cada partición (sección 9.6): regresión lineal, Ridge y Lasso dan una mediana de 0.47. KNN da 0.40, SVR 0.36 y HistGradientBoosting 0.38. Los modelos flexibles no superan al lineal con 16 plantas y 6 zonas. La planta La Loma sale negativa en esta prueba (-0.47), porque su nivel se desplazó entre la primera y la segunda mitad del período. No verificamos la causa.

Sensibilidad (sección 9.4): la mediana va de 0.486 a 0.502 al cambiar la exclusión del arranque o incluir ALMA II.

Lo que estos resultados no permiten decir:
- Que la nubosidad explique algo además de la radiación. La radiación del modelo ya integra sus propias nubes, y la nubosidad diaria no capta cuánto cambia el cielo durante el día.
- Que 0.5 sea el techo del fenómeno. Cambiar de producto de radiación movió la mediana de 0.41 a 0.49, y la tecnología de cada planta no está medida.
- Que valga para cualquier zona. Hay 6 zonas, cinco en el norte y el centro del país y una en los Llanos. Sunnorte, en zona de montaña, sale mucho peor que el resto.

Pendiente:
1. La tecnología de cada planta (seguidor o fija, potencia DC frente a AC). Ninguna de las 16 tiene el montaje confirmado y solo 3 tienen la potencia DC con fuente.
2. Comparar la radiación modelada con radiación medida (estaciones del IDEAM) o satelital.
3. Repetir con datos horarios y con un índice de claridad para la pregunta de las nubes.
4. hyperparameter tuning con validación anidada por zonas, para la versión final.

# Resumen final

Este apartado es el mismo en los cuatro cuadernos del proyecto. Resume todo el trabajo, desde la recolección de los datos hasta los benchmarks, con las métricas,
las decisiones y la razón de cada una. Las cifras salen de la ejecución de los cuadernos el 2 de octubre de 2026.

| Cuaderno | Qué hace |
|---|---|
| `EDA_Corregido(Entregable 3).ipynb` | Limpia y explora la base mundial y produce `dataset_eda_corregido.csv` |
| `Benchmark_Modelos_Base.ipynb` | Modelos base del curso sobre la base mundial |
| `Clima_a_Generacion_Colombia.ipynb` | Construye la base de generación real de Colombia, la explora y prueba modelos preliminares |
| `Benchmark_Colombia.ipynb` | Modelos base del curso sobre la base de Colombia |

## R0. Guion de la presentación, diapositiva por diapositiva

Cada diapositiva dice qué mostrar, qué decir y el número que hay que citar. El detalle y las referencias de cada punto están en las secciones R1 a R15 de este resumen.

**Diapositiva 1. Título y pregunta.**
Qué decir: queríamos predecir qué tan apto es un lugar para una planta solar, a partir del clima, para que sirviera en cualquier parte. Trabajamos con un dataset
mundial publicado en 2026 (Mantilla-Guerra et al., 2026) de 58,978 plantas solares.

**Diapositiva 2. Los datos.**
Mostrar: la tabla de fuentes de R2.
Qué decir: el dataset une cinco fuentes. Global Energy Monitor da las plantas, Global Solar Atlas el potencial solar, ERA5 y NASA POWER el clima, los modelos
de elevación el terreno y OpenStreetMap las carreteras. Son 29 columnas. La variable a predecir es un índice de aptitud solar de 0 a 1 y su versión en tres clases:
Baja, Media y Alta.

**Diapositiva 3. Qué mide de verdad el índice (el primer hallazgo).**
Mostrar: la ecuación del índice de R1.
Qué decir: el paper calcula el índice solo con pendiente, orientación, sombreado y curvatura. El clima no entra. Por eso, aunque quisiéramos, el clima no puede
predecir este índice. Esto cambió el proyecto.

**Diapositiva 4. Pipeline del EDA.**
Mostrar: `fig_pipeline_eda_mundial.png`.
Qué decir: recorrer las cajas. De 58,978 plantas a 57,976 tras quitar filas rotas y duplicados. Se crean variables nuevas, se eligen 15 predictoras y se asigna
cada planta a un bloque del mapa de 5° × 5°.

**Diapositiva 5. Problemas de calidad de datos.**
Mostrar: la tabla de problemas de R3.
Qué decir: 64 plantas sin datos de terreno, todas en la clase Baja. 1,598 duplicados. Distancias a carretera de hasta 2,296 km, que son imposibles. Un 27.7 % de
áreas vacías. Para cada una, la tabla dice qué decisión se tomó y por qué. Los ángulos se pasan a seno y coseno porque 359° y 1° son vecinos.

**Diapositiva 6. El hallazgo central del EDA.**
Mostrar: `fig_mapa_clases_e3.png` o `fig_correlacion_e3.png` del EDA.
Qué decir:
- El índice casi no se relaciona con la pendiente (Spearman -0.07), con la que supuestamente se calcula, y sí con la longitud (+0.58).
- Con solo terreno, un modelo explica R² de 0.18. Con latitud y longitud, 0.91.
- El 94.5 % de la clase Baja viene de un lote de plantas europeas añadido después.

Conclusión: el índice mide sobre todo la región y el lote de procesamiento, no el terreno.

**Diapositiva 7. Por qué validación espacial.**
Mostrar: `fig_pipeline_benchmark_mundial.png`.
Qué decir: si partimos los datos al azar, plantas vecinas quedan una en entrenamiento y otra en prueba, y el modelo "hace trampa" por vecindad (Roberts et al., 2017).
Por eso partimos por bloques de 5° × 5°: un bloque entero va a entrenamiento o a prueba. Además, todo lo que se aprende de los datos (la mediana para rellenar
vacíos, la escala) se ajusta solo con el entrenamiento, dentro de un `Pipeline`, para no filtrar información de la prueba (Kaufman et al., 2012).

**Diapositiva 8. Modelos y métricas.**
Mostrar: las tablas de R6 y R7.
Qué decir: usamos los modelos base del curso. En clasificación, regresión logística, Naive Bayes, KNN y SVM. En regresión, Ridge, Lasso, KNN y SVR. La métrica principal
de clasificación es F1 macro, porque la clase Baja es solo el 3 % y la exactitud premiaría ignorarla. También reportamos AUC, que mide si las probabilidades
ordenan bien sin depender de un umbral, y kappa ponderado, porque las clases tienen orden. En regresión, R², RMSE y MAE.

**Diapositiva 9. hyperparameter tuning sin trampa.**
Mostrar: el esquema de validación anidada de R5.
Qué decir: si elegimos los hiperparámetros mirando los mismos datos con los que reportamos el resultado, el resultado sale inflado (Varma y Simon, 2006).
Por eso usamos validación anidada: un bucle interno elige los hiperparámetros, y otro externo, que nunca participó en la elección, mide.

**Diapositiva 10. Resultados de la base mundial.**
Mostrar: `fig_resumen_comparativa.png` (fila de arriba) y `fig_benchmark_roc_confusion.png`.
Qué decir: el SVM con kernel RBF es el mejor clasificador (F1 macro 0.82, AUC 0.96) y el SVR RBF el mejor en regresión (R² 0.81). KNN va detrás. Los lineales
quedan en F1 0.60 y R² 0.48. La referencia, que siempre predice la clase más común, da F1 0.29.

**Diapositiva 11. Pero, ¿qué aprendieron?**
Mostrar: `fig_benchmark_modelos.png`.
Qué decir:
- Al quitar latitud y longitud, KNN baja de 0.79 a 0.48 en R². Aprende dónde está la planta.
- Con partición aleatoria los resultados salen más altos que con bloques.
- Dejando un continente entero fuera, todos los modelos dan R² negativo, hasta la referencia.

El modelo no sirve fuera de las regiones que vio, y eso confirma el hallazgo del EDA.

**Diapositiva 12. Por qué los lineales quedan bajos.**
Mostrar: la interpretación de R11.
Qué decir: un modelo lineal ajusta un solo plano para todo el mundo, y el índice tiene un nivel distinto en cada región. A eso se le llama no estacionariedad
espacial (Brunsdon et al., 1996). KNN y SVM con kernel ajustan vecindarios locales. Para mejorar se podrían usar bosques aleatorios o XGBoost, modelos espaciales o
stacking. Pero subir el puntaje no hace que el índice mida aptitud.

**Diapositiva 13. La segunda línea: generación real en Colombia.**
Mostrar: `fig_pipeline_clima.png`.
Qué decir: para volver a la idea original del clima, construimos una base propia:
- Generación real hora por hora de 16 plantas colombianas, desde la API pública de XM.
- Clima de cada sitio desde Open-Meteo.
- Coordenadas verificadas con prensa y con las fichas de Global Energy Monitor.

El resultado son 8,589 días planta, de enero de 2024 a febrero de 2026, sin valores vacíos.

**Diapositiva 14. EDA de la base de Colombia.**
Mostrar: `fig_eda_base_colombia.png` y `fig_fc_vs_radiacion.png`.
Qué decir: lo que predecimos es el factor de capacidad, la energía del día dividida entre la máxima posible. El 84 % de su variación es de un día a otro dentro de
cada planta. La radiación es la variable que más se relaciona con él, y nubosidad, temperatura y humedad están muy correlacionadas con ella. Falta la tecnología
de cada planta (si tiene seguidor solar), que es pendiente.

**Diapositiva 15. Resultados en Colombia.**
Mostrar: `fig_resumen_comparativa.png` (fila de abajo) y `fig_benchmark_colombia.png`.
Qué decir:
- Aquí ganan los modelos lineales. Lasso explica R² de 0.47 del sube y baja diario, y el error baja del 20 % al 14 % del factor de capacidad medio.
- Clasificar el día como bajo, medio o alto: la regresión logística acierta 57 de cada 100, con AUC 0.77. El azar daría 33.
- Cada kWh/m² de radiación suma unos 4.7 puntos de factor de capacidad, como predice la física.
- La partición aleatoria vuelve a inflar los resultados.

**Diapositiva 16. Conclusiones.**
Mostrar: R12.
Qué decir: el índice del dataset no mide aptitud desde el clima y no se transfiere entre regiones. Con datos reales, el clima sí explica cerca de la mitad de la
variación diaria de la generación, y casi todo lo aporta la radiación.

**Diapositiva 17. Limitaciones y siguientes pasos.**
Mostrar: R13.
Qué decir: falta la tecnología de cada planta, radiación medida en lugar de modelada, datos por hora, y los modelos de la entrega siguiente (árboles, XGBoost, balanceo con SMOTE).

Preguntas probables y respuesta corta:
- ¿Por qué no partición aleatoria? Porque plantas vecinas se parecen y el resultado sale optimista. Lo mostramos con números en la diapositiva 11.
- ¿Por qué F1 macro y no exactitud? Porque con 75 % de clase Alta, predecir siempre Alta da 75 % de exactitud y no sirve para nada.
- ¿Por qué tan pocas plantas en Colombia? Porque XM no publica coordenadas, y solo usamos plantas cuya ubicación pudimos verificar con fuentes.
- ¿Qué es el AUC? La probabilidad de que el modelo le dé más puntaje a un ejemplo de la clase correcta que a uno de otra clase. 0.5 es azar y 1 es perfecto.

## R1. La pregunta y por qué cambió

La propuesta inicial era predecir la aptitud solar de un sitio a partir del clima, para tener algo útil en cualquier lugar. El dataset del proyecto
(Mantilla-Guerra et al., 2026) trae un índice de aptitud solar (`solar_aptitude`) calculado con la ecuación 1 del paper:
`IAS = 0.40 · pendiente + 0.25 · orientación + 0.20 · sombreado + 0.15 · curvatura`. El clima no entra en esa fórmula, así que no puede predecirse desde el clima.
El EDA confirmó además que el índice depende sobre todo de la región y del lote de procesamiento.

Por eso el trabajo tiene dos líneas:
1. Modelar el índice del dataset y mostrar, con validación espacial, por qué no se transfiere entre regiones.
2. Construir una base propia con la generación real de plantas colombianas y medir cuánto la explica el clima, que es la idea original.

## R2. Recolección de datos

| Fuente | Qué aporta | Acceso |
|---|---|---|
| Dataset de Mantilla-Guerra et al. (2026) | 58,978 plantas FV del mundo, 29 columnas: ubicación, terreno, clima anual, distancia a carretera, potencial FV y el índice | CSV del repositorio de los autores |
| Global Energy Monitor, Global Solar Power Tracker (release de febrero de 2026) | Base de plantas del dataset y fichas públicas con coordenadas | Web de GEM |
| XM, operador del mercado eléctrico colombiano | Generación horaria por planta y capacidad efectiva neta diaria, enero de 2024 a febrero de 2026 | API pública, sin clave |
| Open-Meteo, API histórica (Zippenfenig, 2023) | Radiación, nubosidad, temperatura, humedad y viento por hora, con los productos ERA5 (Hersbach et al., 2020) y `best_match` | API pública, sin clave |

![Pipeline del EDA](fig_pipeline_eda_mundial.png)

![Pipeline de la base de Colombia](fig_pipeline_clima.png)

## R3. EDA y limpieza de la base mundial

Problemas encontrados y cómo se resolvieron:

| Problema | Cifra | Decisión |
|---|---|---|
| Filas sin cobertura de terreno (DEM) | 64 filas, todas en clase Baja (3.4 % de esa clase) | Eliminar |
| Filas sin datos de clima o potencial | 3 filas, más 2 con viento en 0 | Eliminar |
| Duplicados de atributos | 1,598 filas iguales salvo el identificador | Eliminar |
| Orientación con valor -1 en terreno plano | 1,105 filas | Seno y coseno igual a 0 más un indicador `is_flat` |
| Distancia a carretera imposible | 65 filas a más de 100 km, máximo 2,296 km | Pasar a vacío y usar logaritmo |
| Área de la planta | 16,329 vacíos (27.7 %), densidades físicamente imposibles | Excluir |
| Variables que son agrupaciones de otras (`*_type`, `size`, `dt_wind`) | Rangos sin solapamiento | Excluir |

Resultado: 57,976 plantas y 15 predictoras (`dataset_eda_corregido.csv`, 57,976 × 22).

Decisiones y su respaldo:
- Los ángulos (orientación y dirección del viento) se pasan a seno y coseno, porque 359° y 1° son vecinos (Fisher, 1993).
- Se usa correlación de Spearman en lugar de Pearson por las colas pesadas y relaciones no lineales (Hollander et al., 2015).
- Los umbrales de VIF se interpretan con cautela (O'Brien, 2007).
- Se excluyen variables posteriores a la decisión de invertir (capacidad, estado operativo) para no meter información del futuro en el modelo (Kaufman et al., 2012).

Hallazgo central:
- El índice casi no se relaciona con las variables de terreno con las que se calcula: Spearman de -0.066 con la pendiente. Sí se relaciona con la longitud (+0.583).
- Un modelo con solo pendiente, orientación y curvatura explica R² de 0.18. Al añadir latitud y longitud sube a 0.91.
- Dejando fuera una macrorregión, el R² es negativo (-7.3 en Asia y Oceanía, -2.3 en América, -0.1 en Europa, África y Medio Oriente).
- El 94.5 % de la clase Baja viene de un lote europeo que se añadió en la segunda versión del dataset.

La consecuencia es que la validación debe ser espacial. Con partición aleatoria, plantas vecinas quedan a los dos lados y el resultado sale optimista
(Roberts et al., 2017; Ploton et al., 2020). Se definieron 496 bloques de 5° × 5°. En los 5 pliegues, la clase Baja queda entre 2.9 % y 3.8 %.

## R4. Construcción de la base de Colombia

1. Se listaron los 2,213 recursos solares de XM y se quedaron los 27 con al menos 1 MW de capacidad.
2. Se calculó el factor de capacidad diario: `FC = kWh del día / (capacidad efectiva neta × 24 h)`, con la capacidad de ese mismo día.
3. Coordenadas: XM no las publica. Nueve plantas se emparejaron con el dataset del paper y siete con la ficha pública de GEM. Cada emparejamiento lleva
   su motivo y una fuente de prensa o del desarrollador. Por ejemplo, el parque Fundación está en Pivijay (Magdalena). Se descartaron Palmira II, porque
   su ubicación no se confirmó, y Shangri-La, porque sus coordenadas en GEM contradicen su propia ficha.
4. Clima por hora de Open-Meteo en cada coordenada, con dos productos (`era5` y `best_match`). Se resumió por día: radiación diaria, nubosidad media en
   horas de luz, temperatura, humedad y viento. Las horas de XM y de Open-Meteo coinciden sin desfase, y el resumen diario evita problemas de alineación.
5. Filtros con criterios que no miran la variable que se predice: se excluyen los primeros 60 días de las plantas con arranque observado y las plantas con menos de 100 días útiles.

Resultado: `tabla_diaria_colombia.csv`, con 8,589 filas planta-día, 23 columnas, 16 plantas, 6 zonas de cercanía y ningún valor vacío fuera de `fc_ayer`.
Los días sin generación en XM van de 0 a 24 por planta.

EDA de la base:
- El 84 % de la varianza del FC es de un día a otro dentro de cada planta. El 16 % es diferencia entre plantas.
- Dentro de la planta, la radiación diaria explica cerca de la mitad del sube y baja del FC. Cada kWh/m² suma unos 4.7 puntos de FC, casi la proporción física simple.
- Nubosidad, temperatura y humedad están muy correlacionadas con la radiación.
- Pendiente: la tecnología de cada planta (seguidor o fija, potencia DC frente a AC). Ninguna de las 16 la tiene confirmada.

![EDA de la base de Colombia](fig_eda_base_colombia.png)

![FC contra radiación por planta](fig_fc_vs_radiacion.png)

## R5. Diseño experimental común

La partición va siempre primero, y todo lo que se aprende de los datos se ajusta solo con el entrenamiento de cada partición (Kaufman et al., 2012):

| Decisión | Base mundial | Base de Colombia | Respaldo |
|---|---|---|---|
| Unidad de partición | Bloques espaciales de 5° × 5°, 5 pliegues | Zona de plantas a menos de 100 km, más división en el tiempo (pasado para entrenar, futuro para probar) | Roberts et al. (2017); Ploton et al. (2020); Meyer y Pebesma (2021) |
| Imputación y escalado | Dentro de un `Pipeline` de scikit-learn | Dentro de un `Pipeline` | Pedregosa et al. (2011); Kaufman et al. (2012) |
| Nivel de cada planta y cortes de clase | No aplica | Calculados solo con el pasado de cada planta | Kaufman et al. (2012) |
| hyperparameter tuning | Validación anidada: 5 outer folds y 3 internos, todos por bloques | Validación anidada: 6 zonas externas, y en el interno se deja fuera una zona de entrenamiento a la vez | Varma y Simon (2006); Cawley y Talbot (2010) |
| Búsqueda | Grillas pequeñas en escala logarítmica para C y alpha | Igual | Bergstra y Bengio (2012) |
| Probabilidades del SVM lineal | Calibradas con `CalibratedClassifierCV` dentro del entrenamiento | Igual | Niculescu-Mizil y Caruana (2005) |
| SVM con kernel en la base mundial | Submuestra de 8,000 filas por pliegue, porque su costo crece entre cuadrático y cúbico con las filas | No hace falta | Chang y Lin (2011) |

Por qué validación anidada: si se eligen los hiperparámetros con los mismos datos con los que se reporta el resultado, la estimación sale optimista
(Varma y Simon, 2006; Cawley y Talbot, 2010). El bucle interno elige y el externo solo mide.

## R6. Modelos y por qué cada uno

| Modelo | Tarea | Por qué se incluye | Referencia |
|---|---|---|---|
| Referencia (clase mayoritaria, media o nivel de la planta) | Ambas | Mide el piso: cualquier modelo útil debe superarlo | |
| Regresión logística (y versión balanceada) | Clasificación | Lineal, interpretable. La versión con `class_weight="balanced"` compensa la clase Baja | Hastie et al. (2009); He y Garcia (2009) |
| Naive Bayes gaussiano | Clasificación | Modelo probabilístico simple y rápido, línea base del curso | Hastie et al. (2009) |
| KNN | Ambas | No supone forma de la relación. Sensible a la escala, por eso se estandariza | Cover y Hart (1967) |
| SVM lineal y SVM RBF | Clasificación | Margen máximo. El kernel RBF captura relaciones no lineales | Cortes y Vapnik (1995); Chang y Lin (2011) |
| Ridge | Regresión | Lineal con penalización L2, estable con variables correlacionadas | Hoerl y Kennard (1970) |
| Lasso | Regresión | Penalización L1, puede anular variables que sobran | Tibshirani (1996) |
| SVR lineal y SVR RBF | Regresión | Versión de regresión del SVM | Smola y Schölkopf (2004) |
| HistGradientBoosting | Regresión (Colombia, preliminar) | Control flexible basado en árboles | Friedman (2001) |

## R7. Métricas y por qué

| Métrica | Qué mide | Por qué se usa | Referencia |
|---|---|---|---|
| F1 macro | Promedio del F1 de cada clase | Métrica principal en clasificación. Con la clase Baja en 3 %, la exactitud premiaría ignorarla | He y Garcia (2009); Sokolova y Lapalme (2009) |
| Exactitud y exactitud balanceada | Aciertos totales y promedio del recall por clase | La balanceada no depende de la proporción de clases | Sokolova y Lapalme (2009) |
| Precisión y recall macro | Cuánto se acierta al predecir una clase y cuánto se recupera de ella | Separan los dos tipos de error | Sokolova y Lapalme (2009) |
| F1 de la clase Baja | F1 solo de la minoritaria | Es la clase difícil y la más afectada por el lote europeo | He y Garcia (2009) |
| AUC ROC macro uno contra el resto | Qué tan bien ordenan las probabilidades a cada clase frente a las demás, sin umbral | Complementa al F1, que depende del umbral | Fawcett (2006); Hand y Till (2001) |
| Kappa ponderado cuadrático | Acuerdo que castiga más los errores entre clases lejanas | Las clases tienen orden: confundir Baja con Alta es peor que con Media | Cohen (1968) |
| R² | Proporción de la varianza explicada | Métrica principal en regresión. En Colombia se calcula dentro de cada planta | Hastie et al. (2009) |
| RMSE y MAE | Error en las unidades de la variable. RMSE castiga más los errores grandes | Dan la magnitud del error | Hastie et al. (2009) |

## R8. Resultados de la base mundial

Con hiperparámetros por defecto, bloques de 5° y con latitud y longitud (clasificación):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Referencia | 0.285 | 0.748 | 0.333 | 0.249 | 0.333 | 0.000 | 0.500 | 0.000 |
| Regresión logística | 0.599 | 0.768 | 0.559 | 0.695 | 0.559 | 0.523 | 0.865 | 0.367 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.507 | 0.873 | 0.481 |
| Naive Bayes | 0.448 | 0.574 | 0.657 | 0.515 | 0.657 | 0.169 | 0.768 | 0.261 |
| KNN | 0.787 | 0.897 | 0.742 | 0.859 | 0.742 | 0.658 | 0.951 | 0.737 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.812 | 0.896 | 0.772 | 0.869 | 0.772 | 0.747 | 0.958 | 0.751 |

Regresión, mismo escenario:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Referencia | -0.002 | 0.134 | 0.103 |
| Ridge | 0.479 | 0.096 | 0.069 |
| Lasso | 0.478 | 0.096 | 0.069 |
| KNN | 0.790 | 0.061 | 0.037 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.734 | 0.069 | 0.053 |

Los contrastes (F1 macro y R², por defecto):

| Modelo | F1 bloques con lat/lon | F1 bloques sin lat/lon | F1 aleatoria sin lat/lon | R² bloques con lat/lon | R² bloques sin lat/lon | R² aleatoria sin lat/lon |
|---|---|---|---|---|---|---|
| KNN | 0.787 | 0.658 | 0.730 | 0.790 | 0.480 | 0.656 |
| SVM / SVR RBF | 0.812 | 0.617 | 0.680 | 0.734 | 0.427 | 0.560 |
| Regresión logística / Ridge | 0.599 | 0.377 | 0.381 | 0.479 | 0.195 | 0.220 |

Dejando fuera una macrorregión completa, todos los modelos de regresión dan R² negativo, incluida la referencia.

Con hiperparámetros ajustados por validación anidada (5 outer folds y 3 internos, todos por bloques; bloques de 5° y con latitud y longitud):

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | F1 Baja | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|---|
| Regresión logística | 0.600 | 0.768 | 0.560 | 0.694 | 0.560 | 0.525 | 0.865 | 0.368 |
| Regresión logística balanceada | 0.629 | 0.719 | 0.798 | 0.587 | 0.798 | 0.506 | 0.873 | 0.482 |
| Naive Bayes | 0.472 | 0.612 | 0.674 | 0.505 | 0.674 | 0.208 | 0.835 | 0.289 |
| KNN | 0.797 | 0.895 | 0.767 | 0.837 | 0.767 | 0.694 | 0.925 | 0.753 |
| SVM lineal (calibrado) | 0.516 | 0.753 | 0.479 | 0.668 | 0.479 | 0.358 | 0.874 | 0.246 |
| SVM RBF | 0.818 | 0.900 | 0.793 | 0.850 | 0.793 | 0.744 | 0.960 | 0.775 |

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| Ridge | 0.479 | 0.096 | 0.070 |
| Lasso | 0.479 | 0.096 | 0.069 |
| KNN | 0.796 | 0.060 | 0.036 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| SVR RBF | 0.806 | 0.059 | 0.037 |

El ajuste cambia poco a los modelos lineales y más a los no lineales:
- El SVR RBF sube de 0.734 a 0.806 en R². La búsqueda eligió un margen `epsilon` de 0.01 en lugar de 0.1, el valor por defecto. Con un índice que va de 0 a 0.94, un margen de 0.1 ignoraba errores grandes.
- KNN sube de 0.790 a 0.796 en R² y de 0.787 a 0.797 en F1, con 5 vecinos ponderados por distancia.
- Naive Bayes sube de 0.448 a 0.472 en F1 con más suavizado de la varianza.
- Los hiperparámetros elegidos son estables entre pliegues en KNN, Naive Bayes y SVR RBF, y en el SVM RBF el gamma fue `scale` en todos. En la regresión logística, Ridge y Lasso cambian entre pliegues sin que cambie el resultado, lo que indica una superficie plana: la regularización no es el cuello de botella.

![Benchmark mundial](fig_benchmark_modelos.png)

![ROC y matriz de confusión, base mundial](fig_benchmark_roc_confusion.png)

![Efecto del ajuste, base mundial](fig_benchmark_ajuste.png)

## R9. Resultados de la base de Colombia

Hiperparámetros ajustados con validación anidada (zona fuera y futuro de cada planta).

Regresión del FC diario:

| Modelo | R² dentro de planta (mediana) | Peor planta | Mejor planta | RMSE | MAE | MAE relativo |
|---|---|---|---|---|---|---|
| Referencia: nivel de la planta | -0.039 | -0.950 | 0.000 | 0.063 | 0.048 | 0.196 |
| Regresión lineal | 0.466 | -0.469 | 0.619 | 0.046 | 0.035 | 0.141 |
| Ridge | 0.461 | -0.473 | 0.613 | 0.046 | 0.035 | 0.141 |
| Lasso | 0.469 | -0.462 | 0.618 | 0.046 | 0.035 | 0.140 |
| KNN | 0.414 | -0.437 | 0.521 | 0.047 | 0.036 | 0.147 |
| SVR RBF | 0.407 | -0.292 | 0.528 | 0.047 | 0.035 | 0.144 |

Clasificación del día en bajo, medio o alto:

| Modelo | F1 macro | Exactitud | Exactitud bal. | Precisión macro | Recall macro | AUC | Kappa cuad. |
|---|---|---|---|---|---|---|---|
| Referencia | 0.178 | 0.364 | 0.333 | 0.121 | 0.333 | 0.500 | 0.000 |
| Regresión logística | 0.572 | 0.576 | 0.574 | 0.570 | 0.574 | 0.765 | 0.575 |
| Naive Bayes | 0.535 | 0.544 | 0.540 | 0.538 | 0.540 | 0.737 | 0.508 |
| KNN | 0.564 | 0.571 | 0.566 | 0.567 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.558 | 0.559 | 0.539 | 0.559 | 0.751 | 0.553 |
| SVM RBF | 0.572 | 0.576 | 0.570 | 0.580 | 0.570 | 0.759 | 0.568 |

Hiperparámetros elegidos: la búsqueda tomó siempre la opción más suave (KNN con 101 vecinos, SVR con C de 0.1, Ridge con alpha de 100). Con 6 zonas, lo que generaliza
es suavizar mucho. KNN y SVR quedaron en el borde de su grilla.

Contrastes (por defecto): con solo la radiación, el SVR sube de 0.358 a 0.488 en R². Con partición aleatoria, el SVR sube de 0.358 a 0.479 y el SVM RBF de 0.570 a 0.598 en F1.
Las demás variables de clima agregan ruido, y la partición aleatoria infla el resultado.

Prueba de fuentes de clima (cuaderno de clima): la radiación de `best_match` explica más que la de `era5` en 13 de las 16 plantas (mediana de R² de 0.49 frente a 0.41).
Parte del límite del resultado es la calidad de la radiación modelada.

![Benchmark de Colombia](fig_benchmark_colombia.png)

![ROC y matriz de confusión, Colombia](fig_benchmark_colombia_roc_confusion.png)

## R10. Tabla comparativa de todos los modelos

Todos con hiperparámetros ajustados por validación anidada. La mejor fila de cada tarea va primero.

Base mundial, clasificación de la clase de aptitud (bloques de 5°, con latitud y longitud):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. | F1 Baja |
|---|---|---|---|---|---|
| SVM RBF | 0.818 | 0.793 | 0.960 | 0.775 | 0.744 |
| KNN | 0.797 | 0.767 | 0.925 | 0.753 | 0.694 |
| Regresión logística balanceada | 0.629 | 0.798 | 0.873 | 0.482 | 0.506 |
| Regresión logística | 0.600 | 0.560 | 0.865 | 0.368 | 0.525 |
| SVM lineal (calibrado) | 0.516 | 0.479 | 0.874 | 0.246 | 0.358 |
| Naive Bayes | 0.472 | 0.674 | 0.835 | 0.289 | 0.208 |
| Referencia | 0.285 | 0.333 | 0.500 | 0.000 | 0.000 |

Base mundial, regresión del índice:

| Modelo | R² | RMSE | MAE |
|---|---|---|---|
| SVR RBF | 0.806 | 0.059 | 0.037 |
| KNN | 0.796 | 0.060 | 0.036 |
| Ridge | 0.479 | 0.096 | 0.070 |
| SVR lineal | 0.479 | 0.096 | 0.069 |
| Lasso | 0.479 | 0.096 | 0.069 |
| Referencia | -0.002 | 0.134 | 0.103 |

Colombia, regresión del factor de capacidad diario (zona fuera y futuro de cada planta):

| Modelo | R² dentro de planta (mediana) | RMSE | MAE relativo |
|---|---|---|---|
| Lasso | 0.469 | 0.046 | 0.140 |
| Regresión lineal | 0.466 | 0.046 | 0.141 |
| Ridge | 0.461 | 0.046 | 0.141 |
| KNN | 0.414 | 0.047 | 0.147 |
| SVR RBF | 0.407 | 0.047 | 0.144 |
| Referencia | -0.039 | 0.063 | 0.196 |

Colombia, clasificación del día (bajo, medio, alto):

| Modelo | F1 macro | Exactitud bal. | AUC | Kappa cuad. |
|---|---|---|---|---|
| Regresión logística | 0.572 | 0.574 | 0.765 | 0.575 |
| SVM RBF | 0.572 | 0.570 | 0.759 | 0.568 |
| KNN | 0.564 | 0.566 | 0.753 | 0.561 |
| SVM lineal (calibrado) | 0.542 | 0.559 | 0.751 | 0.553 |
| Naive Bayes | 0.535 | 0.540 | 0.737 | 0.508 |
| Referencia | 0.178 | 0.333 | 0.500 | 0.000 |

Lo que cambia entre bases: en la mundial ganan los modelos locales (SVM RBF, SVR RBF, KNN), porque el índice depende de la región. En Colombia ganan los lineales, porque la energía es casi proporcional a la radiación y hay pocas zonas para aprender algo más complejo.

![Comparativa de todos los modelos](fig_resumen_comparativa.png)

## R11. Interpretación: por qué unos modelos quedan bajos y cómo se podría mejorar

### Base mundial

Ridge, Lasso, la regresión logística y el SVM lineal quedan muy por debajo de KNN y del SVM con kernel (R² de 0.48 frente a 0.79; F1 de 0.60 frente a 0.81). La razón es la forma del problema, no la falta de ajuste:

- Un modelo lineal ajusta un único plano para todo el mundo. El índice tiene un nivel distinto en cada región y la relación con el terreno cambia de una región a otra. A eso se le llama no estacionariedad espacial, y un modelo global lineal no puede representarla (Brunsdon et al., 1996).
- KNN y el SVM con kernel ajustan vecindarios locales. Con latitud y longitud entre las variables, aprenden el nivel de cada región. Por eso son los que más caen al quitar latitud y longitud (KNN pasa de 0.79 a 0.48 en R²).
- Lasso no ayuda porque el problema no es que sobren variables. Con 15 variables y 58 mil filas la regularización casi no cambia nada. Con el ajuste, Lasso eligió valores de alpha muy pequeños (de 0.00001 a 0.001), es decir, casi sin penalización, y su R² quedó igual (0.479).
- Naive Bayes es el más bajo porque supone que las variables son independientes dentro de cada clase y con distribución normal. Aquí están correlacionadas (temperatura, humedad, radiación y latitud) y varias no son normales (Hastie et al., 2009).

Qué se podría hacer, según la literatura:
1. Ensambles de árboles: bosques aleatorios (Breiman, 2001) y XGBoost (Chen y Guestrin, 2016). Capturan interacciones y no linealidades sin que haya que especificarlas, y los pide la entrega siguiente. Lo esperable es un resultado parecido al de KNN y SVM con kernel dentro de las regiones conocidas.
2. Modelos espaciales: un bosque aleatorio con distancias o coordenadas como covariables (Hengl et al., 2018), o una regresión geográficamente ponderada, que estima coeficientes distintos en cada lugar y hace explícita la no estacionariedad (Brunsdon et al., 1996).
3. Stacking: combinar las predicciones de varios modelos (lineal, KNN y SVM) con un modelo de segundo nivel entrenado con predicciones fuera de pliegue (Wolpert, 1992).
4. Una advertencia: subir el puntaje dentro de las regiones conocidas no arregla la transferencia entre continentes ni hace que el índice mida aptitud. El área de aplicabilidad (Meyer y Pebesma, 2021) sirve para delimitar dónde una predicción es confiable. Quitar el efecto del lote de procesamiento exigiría recalcular el índice con datos homogéneos, no un mejor modelo.

### Base de Colombia

Aquí pasa lo contrario: los lineales son los mejores (R² de 0.47) y los flexibles quedan por debajo (0.41). La física explica por qué. La energía de una planta es casi proporcional a la radiación que recibe, así que un modelo lineal ya tiene la forma correcta. Con solo 6 zonas, los modelos flexibles aprenden detalles de esas zonas que no se repiten en otras. Por eso la búsqueda de hiperparámetros eligió siempre la opción más suave.

El techo de alrededor de 0.5 viene de los datos, no del modelo:
- La radiación es modelada, no medida en la planta. Cambiar de producto (`era5` a `best_match`) subió la mediana de 0.41 a 0.49. Urraca et al. (2018) evaluaron la radiación de ERA5 contra estaciones y productos satelitales. Su lectura completa está pendiente.
- Los promedios diarios esconden la variación de las nubes dentro del día.
- El nivel de cada planta depende de su tecnología y de recortes o mantenimiento que no medimos, y en algunas plantas cambia con el tiempo.

Qué se podría hacer, según la literatura:
1. Un modelo híbrido físico y estadístico: calcular la radiación en el plano de los paneles y la temperatura de celda con pvlib (Holmgren et al., 2018), con la tecnología real de cada planta, y usar el aprendizaje automático solo para corregir el residuo. Antonanzas et al. (2016) revisan los enfoques físicos, estadísticos e híbridos para pronóstico FV.
2. Mejor radiación: productos satelitales o mediciones en estaciones en lugar de reanálisis.
3. Modelos de efectos mixtos con un intercepto aleatorio por planta (Bates et al., 2015), o un bosque aleatorio de efectos mixtos (Hajjem et al., 2014). Manejan el nivel de cada planta y la dependencia entre días de la misma planta dentro del modelo, en lugar de centrar a mano.
4. Datos horarios y un índice de claridad, para separar el efecto de las nubes del de la geometría solar (Antonanzas et al., 2016).
5. Ensambles de árboles: con 6 zonas probablemente aportan poco, porque los modelos flexibles ya quedaron por debajo del lineal. Con más plantas y zonas podrían ayudar.
6. Completar la tecnología de cada planta, que es lo que más probablemente explica el 16 % de la varianza entre plantas.

## R12. Conclusiones

1. El índice de aptitud del dataset no sirve para predecir aptitud desde el clima. Su fórmula solo usa terreno, y en la práctica depende sobre todo de la región y del lote de procesamiento.
2. Los modelos base del curso predicen bien el índice dentro de las regiones conocidas. El SVM RBF llega a F1 macro de 0.82 y AUC de 0.96, y el SVR RBF a R² de 0.81. Pero lo logran en buena parte reconociendo la ubicación: sin latitud y longitud caen, y dejando una región fuera todos dan R² negativo.
3. Con la validación espacial por bloques los resultados salen más bajos que con partición aleatoria. Esa diferencia es la que justifica la validación espacial.
4. La base de Colombia, construida con generación real de XM y clima de Open-Meteo para 16 plantas, muestra que el clima del día explica cerca de la mitad del sube y baja diario de una planta (R² de 0.47 a 0.49 dentro de planta). Casi todo lo aporta la radiación, y cada kWh/m² suma unos 4.7 puntos de factor de capacidad, como predice la física.
5. En Colombia los modelos lineales son los mejores. El límite está en los datos: radiación modelada, promedios diarios, pocas zonas y la tecnología de cada planta sin medir.
6. El hyperparameter tuning con validación anidada mejora poco los modelos lineales y algo más los no lineales (el SVR RBF mundial de 0.73 a 0.81). En casi todos los casos el cuello de botella son los datos y no los hiperparámetros.

## R13. Limitaciones y pendientes

- La tecnología de cada planta colombiana (seguidor o fija, potencia DC frente a AC) no se ha investigado. Es lo que más probablemente explica el 16 % de la varianza entre plantas.
- El clima de Colombia es modelado, no medido. Falta comparar con radiación medida por estaciones del IDEAM.
- Las grillas de hiperparámetros son pequeñas. En Colombia, KNN y SVR quedaron en el borde de su grilla.
- No se usaron todavía árboles de decisión, bosques aleatorios ni XGBoost, ni balanceo con SMOTE o ADASYN, que pide la entrega siguiente.
- Los SVM con kernel de la base mundial se entrenaron con una submuestra de 8,000 filas.
- Las referencias se verificaron contra Crossref, DataCite o la página de la revista. Antes de citarlas en el informe final hay que leer los textos completos.

## R14. Referencias

- Antonanzas, J., Osorio, N., Escobar, R., Urraca, R. et al. (2016). Review of photovoltaic power forecasting. *Solar Energy, 136*, 78-111. https://doi.org/10.1016/j.solener.2016.06.069
- Bates, D., Mächler, M., Bolker, B. y Walker, S. (2015). Fitting linear mixed-effects models using lme4. *Journal of Statistical Software, 67*(1). https://doi.org/10.18637/jss.v067.i01
- Bergstra, J. y Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281-305. https://jmlr.org/papers/v13/bergstra12a.html
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5-32. https://doi.org/10.1023/A:1010933404324
- Brunsdon, C., Fotheringham, A. S. y Charlton, M. E. (1996). Geographically weighted regression: a method for exploring spatial nonstationarity. *Geographical Analysis, 28*(4), 281-298. https://doi.org/10.1111/j.1538-4632.1996.tb00936.x
- Cawley, G. C. y Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079-2107. https://jmlr.org/papers/v11/cawley10a.html
- Chang, C.-C. y Lin, C.-J. (2011). LIBSVM: a library for support vector machines. *ACM Transactions on Intelligent Systems and Technology, 2*, 1-27. https://doi.org/10.1145/1961189.1961199
- Chen, T. y Guestrin, C. (2016). XGBoost: a scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794. https://doi.org/10.1145/2939672.2939785
- Cohen, J. (1968). Weighted kappa: nominal scale agreement with provision for scaled disagreement or partial credit. *Psychological Bulletin, 70*, 213-220. https://doi.org/10.1037/h0026256
- Cortes, C. y Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*, 273-297. https://doi.org/10.1007/BF00994018
- Cover, T. y Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*, 21-27. https://doi.org/10.1109/TIT.1967.1053964
- Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*, 861-874. https://doi.org/10.1016/j.patrec.2005.10.010
- Fisher, N. I. (1993). *Statistical analysis of circular data*. Cambridge University Press. https://doi.org/10.1017/CBO9780511564345
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*. https://doi.org/10.1214/aos/1013203451
- Global Energy Monitor. (2026). *Global Solar Power Tracker, February 2026 release*. https://globalenergymonitor.org/projects/global-solar-power-tracker
- Hajjem, A., Bellavance, F. y Larocque, D. (2014). Mixed-effects random forest for clustered data. *Journal of Statistical Computation and Simulation, 84*(6), 1313-1328. https://doi.org/10.1080/00949655.2012.741599
- Hand, D. J. y Till, R. J. (2001). A simple generalisation of the area under the ROC curve for multiple class classification problems. *Machine Learning, 45*, 171-186. https://doi.org/10.1023/A:1010920819831
- Hastie, T., Tibshirani, R. y Friedman, J. (2009). *The elements of statistical learning* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7
- He, H. y Garcia, E. A. (2009). Learning from imbalanced data. *IEEE Transactions on Knowledge and Data Engineering, 21*, 1263-1284. https://doi.org/10.1109/TKDE.2008.239
- Hengl, T., Nussbaum, M., Wright, M. N., Heuvelink, G. B. M. et al. (2018). Random forest as a generic framework for predictive modeling of spatial and spatio-temporal variables. *PeerJ, 6*, e5518. https://doi.org/10.7717/peerj.5518
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society, 146*(730), 1999-2049. https://doi.org/10.1002/qj.3803
- Hoerl, A. E. y Kennard, R. W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics, 12*, 55-67. https://doi.org/10.1080/00401706.1970.10488634
- Hollander, M., Wolfe, D. A. y Chicken, E. (2015). *Nonparametric statistical methods*. Wiley. https://doi.org/10.1002/9781119196037
- Holmgren, W. F., Hansen, C. W. y Mikofski, M. A. (2018). pvlib python: a python package for modeling solar energy systems. *Journal of Open Source Software, 3*(29), 884. https://doi.org/10.21105/joss.00884
- Kaufman, S., Rosset, S., Perlich, C. y Stitelman, O. (2012). Leakage in data mining: formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data, 6*(4), 1-21. https://doi.org/10.1145/2382577.2382579
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Niculescu-Mizil, A. y Caruana, R. (2005). Predicting good probabilities with supervised learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625-632. https://doi.org/10.1145/1102351.1102430
- O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. *Quality and Quantity, 41*, 673-690. https://doi.org/10.1007/s11135-006-9018-6
- Pedregosa, F. et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research, 12*, 2825-2830. https://jmlr.org/papers/v12/pedregosa11a.html
- Ploton, P. et al. (2020). Spatial validation reveals poor predictive performance of large-scale ecological mapping models. *Nature Communications, 11*, 4540. https://doi.org/10.1038/s41467-020-18321-y
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Smola, A. J. y Schölkopf, B. (2004). A tutorial on support vector regression. *Statistics and Computing, 14*, 199-222. https://doi.org/10.1023/B:STCO.0000035301.49549.88
- Sokolova, M. y Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing and Management, 45*, 427-437. https://doi.org/10.1016/j.ipm.2009.03.002
- Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society B, 58*, 267-288. https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
- Urraca, R. et al. (2018). Evaluation of global horizontal irradiance estimates from ERA5 and COSMO-REA6 reanalyses using ground and satellite-based data. *Solar Energy, 164*, 339-354. https://doi.org/10.1016/j.solener.2018.02.059
- Varma, S. y Simon, R. (2006). Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics, 7*, 91. https://doi.org/10.1186/1471-2105-7-91
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241-259. https://doi.org/10.1016/S0893-6080(05)80023-1
- XM S.A. E.S.P. (2026). *API pública de datos del mercado de energía mayorista y portal SiMEM*. https://www.simem.co/
- Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [software]. Zenodo. https://doi.org/10.5281/zenodo.7970649

## R15. Terminología

| Término | Qué es | Para qué se usó en este proyecto |
|---|---|---|
| Índice de aptitud solar (IAS, `solar_aptitude`) | Índice de 0 a 1 que el paper calcula con pendiente, orientación, sombreado y curvatura | Variable objetivo de la base mundial, en regresión y en sus tres clases |
| Factor de capacidad (FC) | Energía generada dividida entre la máxima posible a potencia nominal en el mismo tiempo | Variable objetivo de la base de Colombia |
| Capacidad efectiva neta | Potencia máxima que la planta puede entregar a la red, según XM | Denominador del factor de capacidad |
| Potencia DC y AC | Potencia pico de los paneles (DC) y la que sale del inversor (AC) | Su relación es parte de la tecnología pendiente de cada planta |
| Seguidor solar | Estructura que gira los paneles para seguir al sol | Tecnología pendiente; cambia el nivel de FC de cada planta |
| Radiación global horizontal (GHI) | Energía solar que llega a una superficie horizontal, en kWh/m² por día | Principal variable de clima en la base de Colombia |
| Nubosidad | Fracción del cielo cubierta por nubes | Variable de clima; se probó si aporta además de la radiación |
| Reanálisis (ERA5) | Reconstrucción del clima pasado que combina modelo y observaciones | Fuente del clima; se comparó con `best_match` |
| `best_match` | Opción por defecto de Open-Meteo, que combina IFS HRES, ERA5 y ERA5-Land | Fuente de clima que mejor explicó la generación |
| NOCT | Temperatura de operación nominal de una celda, usada para estimarla desde la temperatura del aire | Línea base física con pérdida por temperatura |
| Dato centinela | Valor que en realidad significa "sin dato" (un 0 o un -1) | Se detectaron y se eliminaron o recodificaron en el EDA |
| Fuga de información (data leakage) | Que el modelo use, al entrenar, información que no tendría en la práctica, como los datos de prueba | Se evitó con la partición primero y todo el preprocesamiento dentro del `Pipeline` |
| `Pipeline` | Cadena de pasos de scikit-learn que se ajustan juntos solo con el entrenamiento | Imputación, escalado, calibración y modelo en un solo objeto |
| Imputación por mediana | Rellenar un vacío con la mediana del entrenamiento | 63 vacíos de distancia a carretera en la base mundial |
| Estandarización | Restar la media y dividir entre la desviación estándar | Necesaria para KNN, SVM y modelos regularizados |
| Validación cruzada | Repetir entrenamiento y prueba en varias particiones | Base de toda la evaluación |
| Validación espacial por bloques | Partir por zonas geográficas enteras en vez de filas al azar | Evitar que plantas vecinas queden en entrenamiento y prueba a la vez |
| Partición temporal | Entrenar con el pasado y probar con el futuro | Base de Colombia: el nivel de cada planta sale de su pasado |
| Validación anidada | Bucle interno que elige hiperparámetros y bucle externo que mide | hyperparameter tuning sin inflar el resultado |
| Hiperparámetro | Parámetro que no se aprende de los datos y se fija antes (C, alpha, k) | Se ajustaron con búsqueda en grilla |
| Búsqueda en grilla | Probar todas las combinaciones de una lista de valores | hyperparameter tuning |
| Regularización (L1, L2) | Penalización que limita los coeficientes del modelo | Ridge (L2), Lasso (L1) y el parámetro C de logística y SVM |
| Kernel RBF | Función que permite al SVM separar datos con fronteras no lineales | SVM y SVR con kernel |
| Calibración de probabilidades | Ajustar las salidas del modelo para que se comporten como probabilidades | Probabilidades del SVM lineal, necesarias para el AUC |
| Clase desbalanceada | Clase con muchas menos observaciones que las demás | La clase Baja (3 %) en la base mundial |
| `class_weight="balanced"` | Dar más peso en el entrenamiento a las clases poco frecuentes | Regresión logística balanceada |
| F1 macro | Promedio simple del F1 de cada clase | Métrica principal de clasificación |
| AUC ROC | Probabilidad de que el modelo ordene bien un ejemplo positivo frente a uno negativo | Medir la calidad de las probabilidades sin depender de un umbral |
| Kappa ponderado cuadrático | Acuerdo entre predicción y realidad que castiga más los errores entre clases lejanas | Clases con orden (Baja, Media, Alta; bajo, medio, alto) |
| R² | Proporción de la varianza explicada por el modelo | Métrica principal de regresión |
| RMSE y MAE | Raíz del error cuadrático medio y error absoluto medio | Magnitud del error en unidades de la variable |
| R² dentro de planta | R² calculado con la variable centrada en la media de cada planta | Medir cuánto explica el clima el sube y baja diario, sin el nivel de cada planta |
| Leave-region-out | Entrenar sin una región entera y probar en ella | Mostrar que el modelo del índice no se transfiere entre continentes |
| Bootstrap por planta | Remuestrear plantas, no filas, para obtener intervalos de confianza | Intervalos del R² en el cuaderno de clima |
| Prueba de Wilcoxon | Prueba no paramétrica para diferencias pareadas | Comparar modelos planta por planta en el cuaderno de clima |

## 12. Referencias

Todas verificadas contra Crossref o DataCite el 2 de octubre de 2026. Los BibTeX están en `referencias.bib`. Las entradas sin DOI llevan la URL. Antes de usar una cita en el informe final hay que leer el texto completo.

- Enel Green Power. (2023). *El parque solar Fundación, construido por Enel Green Power en el departamento del Magdalena, inicia su etapa de pruebas*. https://www.enelgreenpower.com/es/medios/press/2023/09/egp-colombia-inicio-etapa-de-pruebas-parque-solar-fundacion
- Faiman, D. (2008). Assessing the outdoor operating temperature of photovoltaic modules. *Progress in Photovoltaics*. https://doi.org/10.1002/pip.813
- Field, C. A. y Welsh, A. H. (2007). Bootstrapping clustered data. *Journal of the Royal Statistical Society B*. https://doi.org/10.1111/j.1467-9868.2007.00593.x
- Friedman, J. H. (2001). Greedy function approximation: a gradient boosting machine. *The Annals of Statistics*. https://doi.org/10.1214/aos/1013203451
- Global Energy Monitor. (2026). *Global Solar Power Tracker, February 2026 release*. https://globalenergymonitor.org/projects/global-solar-power-tracker
- Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society, 146*(730), 1999-2049. https://doi.org/10.1002/qj.3803
- Mantilla-Guerra, A., Mejia-Escobar, C., Azorin-Lopez, J. y Garcia-Rodriguez, J. (2026). Global dataset of solar power plants: multidimensional integration and analysis. *Eng, 7*(7), 343. https://doi.org/10.3390/eng7070343 (preprint arXiv:2603.20601)
- Meyer, H. y Pebesma, E. (2021). Predicting into unknown space? Estimating the area of applicability of spatial prediction models. *Methods in Ecology and Evolution, 12*(9), 1620-1633. https://doi.org/10.1111/2041-210X.13650
- Muñoz-Sabater, J. et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications. *Earth System Science Data, 13*(9), 4349-4383. https://doi.org/10.5194/essd-13-4349-2021
- pv magazine LATAM. (2023, 15 de marzo). *Avanza en Colombia el parque solar Fundación, de 132 MW*. https://www.pv-magazine-latam.com/2023/03/15/avanza-en-colombia-el-parque-solar-fundacion-de-132-mw/
- Roberts, D. R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913-929. https://doi.org/10.1111/ecog.02881
- Skoplaki, E. y Palyvos, J. A. (2009). On the temperature dependence of photovoltaic module electrical performance: a review of efficiency/power correlations. *Solar Energy, 83*(5), 614-624. https://doi.org/10.1016/j.solener.2008.10.008
- Skoplaki, E. y Palyvos, J. A. (2009). Operating temperature of photovoltaic modules: a survey of pertinent correlations. *Renewable Energy, 34*(1), 23-29. https://doi.org/10.1016/j.renene.2008.04.009
- Urraca, R. et al. (2018). Evaluation of global horizontal irradiance estimates from ERA5 and COSMO-REA6 reanalyses using ground and satellite-based data. *Solar Energy, 164*, 339-354. https://doi.org/10.1016/j.solener.2018.02.059
- XM S.A. E.S.P. (2026). *API pública de datos del mercado de energía mayorista y portal SiMEM*. https://www.simem.co/
- Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [software]. Zenodo. https://doi.org/10.5281/zenodo.7970649